# Windows 로컬 청약 RAG 실습 노트북

이 노트북은 `data_home/` 문서를 사용해서 RAG 파이프라인을 실행하는 Windows 로컬 실습용 노트북입니다.

패키지 설치 셀은 넣지 않았습니다. 이미 프로젝트 실행 환경이 준비되어 있다고 가정합니다.

각 단계는 다음 방식으로 실행합니다.

1. 바로 위 Python 셀에서 `QUERY`, `TOP_K`, `LIMIT` 같은 값을 바꿉니다.
2. 바로 아래 Python 실행 셀을 실행합니다.
3. 결과는 cell output에 출력됩니다.

Windows 환경 차이를 줄이기 위해 `%%bash`를 사용하지 않고, 현재 노트북 커널의 Python으로 스크립트를 실행합니다.

## 1. 프로젝트 폴더 경로 설정

`PROJECT_DIR`만 본인 컴퓨터의 프로젝트 폴더 경로로 고치세요.

Windows 경로 앞에는 `r`을 붙입니다.


In [1]:
from pathlib import Path
import os
import sys

# 예: r"C:\sct-project-graphrag"
PROJECT_DIR = Path(r"D:\snu_python\graphrag")

REQUIRED_MARKERS = ["pyproject.toml", "scripts_home", "data_home"]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}")

missing = [marker for marker in REQUIRED_MARKERS if not (PROJECT_DIR / marker).exists()]
if missing:
    raise RuntimeError(
        f"프로젝트 폴더로 보이지 않습니다: {PROJECT_DIR}\n"
        f"없는 항목: {missing}\n"
        "zip이 한 겹 더 들어가서 풀렸는지 확인하고 PROJECT_DIR을 안쪽 폴더로 고치세요."
    )

os.chdir(PROJECT_DIR)
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)

print("프로젝트 폴더:", PROJECT_DIR)
print("현재 작업 폴더:", Path.cwd())
print("노트북 Python:", sys.executable)


프로젝트 폴더: D:\snu_python\graphrag
현재 작업 폴더: d:\snu_python\graphrag
노트북 Python: c:\Users\Dell\miniforge3\envs\openai\python.exe


## 2. API 환경변수 설정

LLM을 호출하는 단계에서 사용합니다.

raw BM25 검색만 실행할 때는 API key가 없어도 됩니다.


In [2]:
import getpass
import os

api_key = getpass.getpass("조별 API key를 입력하세요. raw 검색만 할 경우 Enter를 눌러 건너뛰어도 됩니다: ")

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key

os.environ["BASE_URL"] = "ldi.snu.ac.kr:30000"
os.environ["OPENAI_MODEL"] = "gpt-5.4-mini"
os.environ["OPENAI_REASONING_EFFORT"] = "low"
os.environ["OPENAI_EMBEDDING_MODEL"] = "text-embedding-3-small"

print("환경변수 설정 완료")
print("BASE_URL:", os.environ["BASE_URL"])
print("OPENAI_MODEL:", os.environ["OPENAI_MODEL"])


환경변수 설정 완료
BASE_URL: ldi.snu.ac.kr:30000
OPENAI_MODEL: gpt-5.4-mini


## 3. 실행 helper

Windows 경로, 출력 buffering, 한글 인코딩 문제를 줄이기 위해 모든 스크립트를 아래 helper로 실행합니다.


In [3]:
import subprocess
import sys
import os


def run_script(script_name: str, *args: object) -> None:
    script_path = PROJECT_DIR / script_name
    command = [sys.executable, "-u", str(script_path), *[str(arg) for arg in args]]
    print("$", " ".join(command), flush=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"

    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


print("실행 helper 준비 완료")


실행 helper 준비 완료


## 4. 공통 파일 경로

필요하면 여기에서 `data_home`을 다른 데이터 폴더로 바꾸면 됩니다.

In [4]:
DATA_DIR = "data_home"

RAW_PAGES = f"{DATA_DIR}/processed/pdf_pages.jsonl"
TOPICS = f"{DATA_DIR}/processed/guide_topics.jsonl"
TOPIC_INDEX = f"{DATA_DIR}/indexes/topic_embedding_index.npz"
TOPIC_METADATA = f"{DATA_DIR}/indexes/topic_embedding_metadata.json"
TOPIC_FEATURES = f"{DATA_DIR}/processed/topic_features.jsonl"
TOPIC_GRAPH = f"{DATA_DIR}/indexes/topic_graph_with_similarity.json"

print("RAW_PAGES:", RAW_PAGES)
print("TOPICS:", TOPICS)


RAW_PAGES: data_home/processed/pdf_pages.jsonl
TOPICS: data_home/processed/guide_topics.jsonl


## 5. Raw text BM25 검색

원문 텍스트에서 바로 키워드 검색을 합니다.

API를 사용하지 않습니다.


In [5]:
QUERY = "분양가상한제가 적용되는 지역에서 특별공급 신청 자격은 어떻게 확인해야 하나?"
TOP_K = 3

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)

QUERY: 분양가상한제가 적용되는 지역에서 특별공급 신청 자격은 어떻게 확인해야 하나?
TOP_K: 3


In [6]:
run_script(
    "scripts_home/06_bm25_raw_text_demo.py",
    QUERY,
    "--input", RAW_PAGES,
    "--top-k", TOP_K,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\06_bm25_raw_text_demo.py 분양가상한제가 적용되는 지역에서 특별공급 신청 자격은 어떻게 확인해야 하나? --input data_home/processed/pdf_pages.jsonl --top-k 3
1. score=12.898 file=주택청약 FAQ.pdf chunk=30
   20 Q236. 입주자모집공고일 현재는 귀화하였으나 귀화 전 외국인으로서 근로소득 또는  사업소득으로 소득세를 납부한 내역이 있는 경우, 소득세 납부사실을 인정할 수  있는지? ·················································································································120 마. 소
2. score=12.563 file=주택청약 FAQ.pdf chunk=24
   요? ·······························································································101 Q189. 외국인 배우자가 있는 한국인의 경우에는 특별공급 신청이 가능한가요? ················102 Q190. 외국인 부부인 경우 신혼부부 특별공급신청이 가능한가요? ·················
3. score=10.719 file=주택청약 FAQ.pdf chunk=41
   하는 경우에는 소득은 어떻게 신장하나요? ·······165 자. 근로소득과 사업소득이 동시에 발생하는 경우·····························································166 Q333. 근로소득과 사업소득이 동시 발생할 경우 월평균소득산정? (근로소득 + 사업소득) 166 Q334. 계속적인 근로소득자이나 최근 사업장을 개업

## 6. Raw text BM25 RAG 답변

원문 검색 결과를 LLM에게 넣어 답변을 생성합니다.

API를 사용합니다.


In [7]:
QUERY = "무주택 기간은 어느 시점부터 인정되며, 어떤 자료로 확인해야 하나?"
TOP_K = 5
MAX_CONTEXT_CHARS = 7000

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)

QUERY: 무주택 기간은 어느 시점부터 인정되며, 어떤 자료로 확인해야 하나?
TOP_K: 5


In [8]:
run_script(
    "scripts_home/09_bm25_raw_rag_answer.py",
    QUERY,
    "--input", RAW_PAGES,
    "--top-k", TOP_K,
    "--max-context-chars", MAX_CONTEXT_CHARS,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\09_bm25_raw_rag_answer.py 무주택 기간은 어느 시점부터 인정되며, 어떤 자료로 확인해야 하나? --input data_home/processed/pdf_pages.jsonl --top-k 5 --max-context-chars 7000
## Retrieved Chunks
1. score=14.522 file=주택청약 FAQ.pdf chunk=125
   .       따라서 주택을 소유하고 있는 직계존속(또는 그 배우자)이 만60세 이상이라는 사유로  제53조제6호가 적용되어 무주택으로 인정받는 경우에는 부양가족에서 제외되어야  하나, 직계존속이 소유한 주택이 제53조제6호를 제외한 다른 사유로 무주택으로 인정 받는 경우에는 부양가족에 포함될 수 있으므로 직계존속이 무주택으로 인정되는 소형 저가주택을 소유하고 있고 세대 내에서 다른 주택을 
2. score=13.094 file=주택청약 FAQ.pdf chunk=99
   주택 무주택 규정 미적용  소형저가 + 제53조 제5호 2  5호 및 소형저가 무주택 규정 미적용    입주자모집공고일 현재 소형저가주택 또는 제53조 각 호의 주택을 기 처분한 경우        (1) 입주자모집공고일 현재 소형저가주택 기 매각 + 제53조 각 호(5호 제외) 기 매각            입주자모집공고일 현재 무주택 인정, 소형저가주택 및 제53조 각 호(5호 제외)의  주
3. score=8.876 file=청약홈 FAQ - 03.청약가점제 chunk=7
   년인 세대주로, 모친은 만68세이며 3채의 아파트로 임대사업을 하고 있습니다. 본인은 무주택자에 해당하나요? A. 분양주택에 가점제 청약 시 무주택자에 해당합니다. 만60세 이상인 직계존속이 주택을 소유한 경우 청약신청자를 무주택자로 간주하므로 모친이 여러채의 주택을 소유하더라도 신청자 본인

## 7. 청약 topic 구조화 추출

원문 문서를 읽고, 사용자가 물어볼 만한 쟁점과 판단 절차를 구조화합니다.

API를 사용합니다.

처음 테스트할 때는 `LIMIT`을 작게 두는 것이 좋습니다.

In [9]:
LIMIT = 5
CONCURRENCY = 2

print("LIMIT:", LIMIT)
print("CONCURRENCY:", CONCURRENCY)
print("OUTPUT:", TOPICS)


LIMIT: 5
CONCURRENCY: 2
OUTPUT: data_home/processed/guide_topics.jsonl


In [10]:
run_script(
    "scripts_home/04_extract_guide_topics.py",
    RAW_PAGES,
    TOPICS,
    "--limit", LIMIT,
    "--concurrency", CONCURRENCY,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\04_extract_guide_topics.py data_home/processed/pdf_pages.jsonl data_home/processed/guide_topics.jsonl --limit 5 --concurrency 2
[info] 입력에서 고유 문서 5건 추출
[info] 기존 출력에서 처리 완료 4건 확인 → 건너뜀
[info] 총 1건 처리 시작 (model=gpt-5.4-mini, reasoning_effort=low, concurrency=2, num_ctx=16384)
[ok]   (1/1) 청약홈 FAQ - 02.청약통장 — 공통, 무주택자, 유주택자 / 청약통장, 순위자격, 변경, 명의변경, 청약신청 / topic 7개
[info] 완료 → data_home\processed\guide_topics.jsonl


## 8. Topic BM25 검색

04번에서 구조화한 topic을 검색합니다.

API를 사용하지 않습니다.


In [11]:
QUERY = "가점제 점수 산정과 무주택 기간 인정 기준"
TOP_K = 5

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)

QUERY: 가점제 점수 산정과 무주택 기간 인정 기준
TOP_K: 5


In [12]:
run_script(
    "scripts_home/07_bm25_topic_demo.py",
    QUERY,
    "--input", TOPICS,
    "--top-k", TOP_K,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\07_bm25_topic_demo.py 가점제 점수 산정과 무주택 기간 인정 기준 --input data_home/processed/guide_topics.jsonl --top-k 5
1. score=10.079 file=주택청약 FAQ.pdf topic=4
   질문: 주택을 소유했다고 보지 않는 예외에 해당하면 무주택으로 인정되나요?
   결과: 요건을 충족하면 해당 주택을 소유하지 않은 것으로 보아 무주택 자격을 인정받을 수 있다.
2. score=9.561 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=10
   질문: 민영주택 가점제는 어떤 항목으로 계산되나요?
   결과: 가점 항목을 바탕으로 민영주택 청약가점을 확인할 수 있다.
3. score=6.645 file=주택청약 FAQ.pdf topic=5
   질문: 주택을 처분했거나 분양권을 취득한 시점에 따라 무주택기간은 어떻게 계산하나요?
   결과: 무주택기간이 산정되어 가점 또는 자격 판단에 반영된다.
4. score=6.596 file=주택청약 FAQ.pdf topic=8
   질문: 당첨되면 당첨자로 관리되거나 재당첨 제한을 받나요?
   결과: 당첨자 관리 등록 및 재당첨 제한 적용 여부가 결정된다.
5. score=6.231 file=청약홈 FAQ - 03.청약가점제 topic=3
   질문: 청약가점제에서 무주택기간과 무주택자 여부는 어떻게 판단하나요?
   결과: 무주택자 해당 여부와 무주택기간이 산정되어 가점제 점수에 반영된다.


## 9. Topic embedding index 생성

구조화된 topic을 embedding index로 만듭니다.

API를 사용합니다.


In [13]:
LIMIT = 50
BATCH_SIZE = 64

print("LIMIT:", LIMIT)
print("BATCH_SIZE:", BATCH_SIZE)
print("INDEX:", TOPIC_INDEX)


LIMIT: 50
BATCH_SIZE: 64
INDEX: data_home/indexes/topic_embedding_index.npz


In [14]:
run_script(
    "scripts_home/05_build_topic_embedding_index.py",
    "--input", TOPICS,
    "--index", TOPIC_INDEX,
    "--metadata", TOPIC_METADATA,
    "--limit", LIMIT,
    "--batch-size", BATCH_SIZE,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\05_build_topic_embedding_index.py --input data_home/processed/guide_topics.jsonl --index data_home/indexes/topic_embedding_index.npz --metadata data_home/indexes/topic_embedding_metadata.json --limit 50 --batch-size 64
topics=48 dim=1536
data_home\indexes\topic_embedding_index.npz
data_home\indexes\topic_embedding_metadata.json


## 10. Dense topic 검색

질문을 embedding으로 바꾸고, topic embedding index에서 의미 검색을 합니다.

API를 사용합니다.


In [15]:
QUERY = "특별공급으로 당첨된 후 부적격으로 취소되는 사유와 확인 방법"
TOP_K = 5

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)

QUERY: 특별공급으로 당첨된 후 부적격으로 취소되는 사유와 확인 방법
TOP_K: 5


In [16]:
run_script(
    "scripts_home/08_dense_topic_search.py",
    QUERY,
    "--index", TOPIC_INDEX,
    "--metadata", TOPIC_METADATA,
    "--top-k", TOP_K,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\08_dense_topic_search.py 특별공급으로 당첨된 후 부적격으로 취소되는 사유와 확인 방법 --index data_home/indexes/topic_embedding_index.npz --metadata data_home/indexes/topic_embedding_metadata.json --top-k 5
1. score=0.445 file=주택청약 FAQ.pdf topic=8
   작업/질문: 당첨되면 당첨자로 관리되거나 재당첨 제한을 받나요?
   결과: 당첨자 관리 등록 및 재당첨 제한 적용 여부가 결정된다.
2. score=0.432 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=19
   작업/질문: 당첨됐는데 건설사에서 부적격당첨자라고 합니다. 왜 그런가요?
   결과: 실제 자격과 다르면 당첨이 취소되고 부적격당첨자로 처리될 수 있다.
3. score=0.402 file=주택공급에 관한 규칙.pdf topic=3
   작업/질문: 청약할 때 어떤 서류를 제출해야 하나요?
   결과: 필수 서류가 갖춰지면 청약 신청과 자격 확인이 진행된다.
4. score=0.393 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=11
   작업/질문: 외국인도 민영주택 특별공급에 청약할 수 있나요?
   결과: 예외 사유가 없으면 외국인은 해당 공급유형에 청약할 수 없다.
5. score=0.384 file=주택청약 FAQ.pdf topic=7
   작업/질문: 특별공급 중 신혼부부나 생애최초 특별공급 자격은 어떻게 판단하나요?
   결과: 요건 충족 시 신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부가 결정된다.


## 11. Topic BM25 + raw text RAG 답변

구조화 topic 검색 결과와 원문 context를 함께 사용해 답변합니다.

API를 사용합니다.


In [17]:
QUERY = "특별공급 신청 자격과 가점제 점수 산정 기준을 함께 확인하려면 어디를 봐야 하나?"
TOP_K = 4
RAW_CONTEXT = "same-document"

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("RAW_CONTEXT:", RAW_CONTEXT)

QUERY: 특별공급 신청 자격과 가점제 점수 산정 기준을 함께 확인하려면 어디를 봐야 하나?
TOP_K: 4
RAW_CONTEXT: same-document


In [18]:
run_script(
    "scripts_home/10_bm25_topic_rag_answer.py",
    QUERY,
    "--topics", TOPICS,
    "--pages", RAW_PAGES,
    "--top-k", TOP_K,
    "--raw-context", RAW_CONTEXT,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\10_bm25_topic_rag_answer.py 특별공급 신청 자격과 가점제 점수 산정 기준을 함께 확인하려면 어디를 봐야 하나? --topics data_home/processed/guide_topics.jsonl --pages data_home/processed/pdf_pages.jsonl --top-k 4 --raw-context same-document
## Retrieved Topics
1. score=9.606 file=주택청약 FAQ.pdf topic=7
   질문: 특별공급 중 신혼부부나 생애최초 특별공급 자격은 어떻게 판단하나요?
   결과: 요건 충족 시 신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부가 결정된다.
2. score=8.702 file=청약홈 FAQ - 03.청약가점제 topic=1
   질문: 청약가점제 적용비율은 면적과 지역에 따라 어떻게 달라지나요?
   결과: 해당 주택 유형에서 가점제와 추첨제의 적용 비율을 확인할 수 있다.
3. score=7.269 file=청약홈 FAQ - 03.청약가점제 topic=3
   질문: 청약가점제에서 무주택기간과 무주택자 여부는 어떻게 판단하나요?
   결과: 무주택자 해당 여부와 무주택기간이 산정되어 가점제 점수에 반영된다.
4. score=6.992 file=청약홈 FAQ - 03.청약가점제 topic=0
   질문: 청약가점제란 무엇이며 당첨자는 어떻게 선정되나요?
   결과: 가점제 점수에 따라 가점제 배정 물량에서 당첨 여부가 결정된다.

## Added Raw Chunks
R1. score=0.000 file=주택청약 FAQ.pdf chunk=0
    [page 1] [추가 정정]  [page 2] 본 FAQ는 2022년 7월 기준으로 작성되었으며, 관련 법령의 개정, 법령해석  변경 등에 따라 변경될 수 

## 12. Search agent demo

agent가 필요한 검색어를 스스로 고르고, 여러 번 검색한 뒤 답변합니다.

API를 사용합니다.


In [19]:
QUERY = "특별공급으로 신청했는데 가점제 점수 산정 기준과 무주택 기간 인정 범위, 그리고 부적격으로 당첨 취소되는 경우는 어디에서 확인해야 하나?"
TOP_K = 4
MAX_STEPS = 4

print("QUERY:", QUERY)
print("TOP_K:", TOP_K)
print("MAX_STEPS:", MAX_STEPS)

QUERY: 특별공급으로 신청했는데 가점제 점수 산정 기준과 무주택 기간 인정 범위, 그리고 부적격으로 당첨 취소되는 경우는 어디에서 확인해야 하나?
TOP_K: 4
MAX_STEPS: 4


In [20]:
run_script(
    "scripts_home/15_search_agent_demo.py",
    QUERY,
    "--topics", TOPICS,
    "--pages", RAW_PAGES,
    "--top-k", TOP_K,
    "--max-steps", MAX_STEPS,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\15_search_agent_demo.py 특별공급으로 신청했는데 가점제 점수 산정 기준과 무주택 기간 인정 범위, 그리고 부적격으로 당첨 취소되는 경우는 어디에서 확인해야 하나? --topics data_home/processed/guide_topics.jsonl --pages data_home/processed/pdf_pages.jsonl --top-k 4 --max-steps 4
## User Query
특별공급으로 신청했는데 가점제 점수 산정 기준과 무주택 기간 인정 범위, 그리고 부적격으로 당첨 취소되는 경우는 어디에서 확인해야 하나?

## Step 1: search_topics
reason: 특별공급 관련 가점 산정, 무주택 기간 인정, 부적격 당첨 취소 사유의 적용 기준과 판단 절차가 함께 필요하므로 먼저 쟁점 구조를 검색해 근거 범위를 파악하려는 검색입니다.
query: 특별공급 가점제 점수 산정 기준 무주택 기간 인정 범위 부적격 당첨 취소 확인 기준
S1-1. topic score=16.556 file=주택청약 FAQ.pdf topic=8
   질문: 당첨되면 당첨자로 관리되거나 재당첨 제한을 받나요?
   결과: 당첨자 관리 등록 및 재당첨 제한 적용 여부가 결정된다.
S1-2. topic score=12.916 file=주택청약 FAQ.pdf topic=4
   질문: 주택을 소유했다고 보지 않는 예외에 해당하면 무주택으로 인정되나요?
   결과: 요건을 충족하면 해당 주택을 소유하지 않은 것으로 보아 무주택 자격을 인정받을 수 있다.
S1-3. topic score=10.099 file=청약홈 FAQ - 03.청약가점제 topic=3
   질문: 청약가점제에서 무주택기간과 무주택자 여부는 어떻게 판단하나요?
   결과: 무주택자 해당 여부와 무주택기간이 산정되어 가점제 점수에 반

## 13. Graph feature 추출

Graph RAG까지 볼 때만 실행합니다.

API를 사용합니다.


In [21]:
LIMIT = 30
CONCURRENCY = 2

print("LIMIT:", LIMIT)
print("OUTPUT:", TOPIC_FEATURES)


LIMIT: 30
OUTPUT: data_home/processed/topic_features.jsonl


In [22]:
run_script(
    "scripts_home/11_extract_graph_features.py",
    "--input", TOPICS,
    "--output", TOPIC_FEATURES,
    "--limit", LIMIT,
    "--concurrency", CONCURRENCY,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\11_extract_graph_features.py --input data_home/processed/guide_topics.jsonl --output data_home/processed/topic_features.jsonl --limit 30 --concurrency 2
[info] topic records=30 done=0 todo=30
[ok] (1/30) 주택공급에 관한 규칙.pdf::0
[ok] (2/30) 주택공급에 관한 규칙.pdf::1
[ok] (3/30) 주택공급에 관한 규칙.pdf::2
[ok] (4/30) 주택공급에 관한 규칙.pdf::3
[ok] (5/30) 주택공급에 관한 규칙.pdf::4
[ok] (6/30) 주택청약 FAQ.pdf::0
[ok] (7/30) 주택청약 FAQ.pdf::1
[ok] (8/30) 주택청약 FAQ.pdf::2
[ok] (9/30) 주택청약 FAQ.pdf::4
[ok] (10/30) 주택청약 FAQ.pdf::3
[ok] (11/30) 주택청약 FAQ.pdf::5
[ok] (12/30) 주택청약 FAQ.pdf::6
[ok] (13/30) 주택청약 FAQ.pdf::7
[ok] (14/30) 청약홈 FAQ - 01.청약이 잘 안되세요?::0
[ok] (15/30) 주택청약 FAQ.pdf::8
[ok] (16/30) 청약홈 FAQ - 01.청약이 잘 안되세요?::1
[ok] (17/30) 청약홈 FAQ - 01.청약이 잘 안되세요?::2
[ok] (18/30) 청약홈 FAQ - 01.청약이 잘 안되세요?::3
[ok] (19/30) 청약홈 FAQ - 01.청약이 잘 안되세요?::4
[warn] 청약홈 FAQ - 01.청약이 잘 안되세요?::5 attempt 1/3 failed: BadRequestError: Error code: 400 - {'error': {'

## 14. Topic graph 생성

11번 결과를 graph JSON으로 만듭니다.

`ADD_SIMILARITY`가 `False`이면 API를 사용하지 않습니다. `True`이면 similarity edge 생성을 위해 embedding API를 사용합니다.


In [23]:
ADD_SIMILARITY = False

print("ADD_SIMILARITY:", ADD_SIMILARITY)
print("GRAPH:", TOPIC_GRAPH)


ADD_SIMILARITY: False
GRAPH: data_home/indexes/topic_graph_with_similarity.json


In [24]:
args = [
    "--input", TOPIC_FEATURES,
    "--output", TOPIC_GRAPH,
]
if ADD_SIMILARITY:
    args.append("--add-similarity")

run_script("scripts_home/12_build_topic_graph.py", *args)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\12_build_topic_graph.py --input data_home/processed/topic_features.jsonl --output data_home/indexes/topic_graph_with_similarity.json
nodes=370 edges=393
data_home\indexes\topic_graph_with_similarity.json


## 15. Graph viewer HTML 생성

브라우저에서 볼 수 있는 graph HTML 파일을 만듭니다.

API를 사용하지 않습니다.


In [25]:
VIEWER_OUTPUT = "results/housing_graph_viewer_sigma.html"

print("VIEWER_OUTPUT:", VIEWER_OUTPUT)

VIEWER_OUTPUT: results/housing_graph_viewer_sigma.html


In [26]:
run_script(
    "scripts_home/13_export_sigma_viewer.py",
    "--input", TOPIC_GRAPH,
    "--output", VIEWER_OUTPUT,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\13_export_sigma_viewer.py --input data_home/indexes/topic_graph_with_similarity.json --output results/housing_graph_viewer_sigma.html
results\housing_graph_viewer_sigma.html
nodes=370 edges=393


## 16. Graph RAG 답변

Graph에서 관련 topic을 찾고 LLM으로 답변합니다.

API를 사용합니다.


In [27]:
QUERY = "특별공급 자격, 가점제 산정, 당첨 취소가 서로 어떻게 연결되어 있나?"
TOP_SEEDS = 8
MAX_TOPICS = 30

print("QUERY:", QUERY)
print("TOP_SEEDS:", TOP_SEEDS)
print("MAX_TOPICS:", MAX_TOPICS)

QUERY: 특별공급 자격, 가점제 산정, 당첨 취소가 서로 어떻게 연결되어 있나?
TOP_SEEDS: 8
MAX_TOPICS: 30


In [28]:
run_script(
    "scripts_home/14_graph_rag_answer.py",
    QUERY,
    "--graph", TOPIC_GRAPH,
    "--top-seeds", TOP_SEEDS,
    "--max-topics", MAX_TOPICS,
)

$ c:\Users\Dell\miniforge3\envs\openai\python.exe -u D:\snu_python\graphrag\scripts_home\14_graph_rag_answer.py 특별공급 자격, 가점제 산정, 당첨 취소가 서로 어떻게 연결되어 있나? --graph data_home/indexes/topic_graph_with_similarity.json --top-seeds 8 --max-topics 30
## Seed Phrase Nodes
1. score=10.40 type=Setting label=가점제 당첨·예비입주자 기준
2. score=10.11 type=Task label=신혼부부 특별공급 자격 판단
3. score=10.11 type=Task label=생애최초 특별공급 자격 판단
4. score=9.13 type=Task label=가점제 또는 일반공급에서 무주택기간 산정
5. score=8.81 type=Task label=특별공급 횟수 제한과 가점제 재당첨 제한 해당 여부를 확인
6. score=7.05 type=Screen label=민영주택 가점제
7. score=6.87 type=Screen label=가점제 재당첨 제한
8. score=5.92 type=Screen label=신혼부부 특별공급

## Graph Retrieval Summary
expanded_phrase_nodes=8
retrieved_topics=4
analysis_mode=overview
accepted_topics=0
rejected_topics=0
other_topics=4

## Representative Topics
Top graph matches:
  [C1] file=주택청약 FAQ.pdf topic=7 outcome=신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부 결정 score=26.14
       작업/질문: 특별공급 중 신혼부부나 생애최초 특별공급 자격은 어떻게 판단하나요?
       결론: 요건 충족 시 신혼부

## 17. 파이프라인 비교 · 질문 / 성능지표 / md 저장

위 5~16절은 한 파이프라인씩 단계별로 실행하는 walkthrough였습니다.
이 절에서는 같은 질문을 **네 파이프라인에 한 번에** 넣고, 답변과 성능지표를 모아
**질문별 / 파이프라인별 최종 결과를 하나의 md 파일로 저장**합니다.

| | 파이프라인 | 실행 스크립트 | 입력 |
|---|---|---|---|
| 1 | **Chunk BM25** (baseline) | `scripts_home/09_bm25_raw_rag_answer.py` | `RAW_PAGES` |
| 2 | **Topic-based** (structural) | `scripts_home/10_bm25_topic_rag_answer.py --raw-context same-document` | `TOPICS` (+ `RAW_PAGES`) |
| 3 | **Graph RAG** (relational) | `scripts_home/14_graph_rag_answer.py --top-seeds --max-topics` | `TOPIC_GRAPH` |
| 4 | **Search Agent** (multi-step) | `scripts_home/15_search_agent_demo.py --max-steps` | `TOPICS` (+ `RAW_PAGES`) |

> 파이프라인 4는 agent가 스스로 여러 번 검색한 뒤 답하므로 1~3보다 느리고 API 호출이 많습니다.
> 일부만 비교하려면 `compare(..., which=(1, 3))`처럼 지정하세요. (06/07/08 같은 순수 검색 데모는 `## Answer`를 내지 않아 답변 비교에는 넣지 않았습니다.)

위 1·8절에서 정의한 `PROJECT_DIR`, `RAW_PAGES`, `TOPICS`, `TOPIC_GRAPH` 변수와
2절의 API 환경변수를 그대로 사용합니다. (먼저 1·2·8절을 실행해 두세요. P3까지 보려면 13~14절로 `TOPIC_GRAPH`가 만들어져 있어야 합니다.)

In [29]:
import subprocess
import time

PIPELINE_TITLES = {
    1: "파이프라인 1 · Chunk BM25 (baseline)",
    2: "파이프라인 2 · Topic-based (structural)",
    3: "파이프라인 3 · Graph RAG (relational)",
    4: "파이프라인 4 · Search Agent (multi-step)",
}


def _run_capture(script_name, *args, timeout=900):
    """scripts를 subprocess로 실행하고 (returncode, stdout, elapsed)를 돌려줍니다.
    run_script와 달리 출력을 capture하고, 실패해도 예외를 던지지 않습니다."""
    command = [sys.executable, "-u", str(PROJECT_DIR / script_name), *[str(a) for a in args]]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    t0 = time.time()
    proc = subprocess.run(
        command, cwd=PROJECT_DIR, capture_output=True, text=True,
        encoding="utf-8", errors="replace", env=env, timeout=timeout,
    )
    elapsed = time.time() - t0
    out = proc.stdout or ""
    if proc.returncode != 0:
        out += "\n\n[stderr]\n" + (proc.stderr or "")
    return proc.returncode, out, elapsed


def _split_answer(stdout):
    """(retrieval_log, answer_text)로 분리. '## Answer'가 단독 헤더 줄로
    처음 나오는 지점에서 자른다. 답변 본문의 ##/### 헤더에 오작동하지 않도록 줄 단위로 매칭."""
    lines = stdout.splitlines()
    for i, line in enumerate(lines):
        if line.strip() == "## Answer":
            log = "\n".join(lines[:i]).strip()
            answer = "\n".join(lines[i + 1:]).strip()
            return log, answer
    # '## Answer' 헤더가 없으면 전체 출력을 답변으로 본다.
    return "", stdout.strip()


def _run_pipeline(n, query, top_k_raw, top_k_topic, top_seeds, max_topics, top_k_agent, max_steps):
    if n == 1:
        return _run_capture(
            "scripts_home/09_bm25_raw_rag_answer.py", query,
            "--input", RAW_PAGES, "--top-k", top_k_raw,
        )
    if n == 2:
        return _run_capture(
            "scripts_home/10_bm25_topic_rag_answer.py", query,
            "--topics", TOPICS, "--pages", RAW_PAGES,
            "--top-k", top_k_topic, "--raw-context", "same-document",
        )
    if n == 3:
        return _run_capture(
            "scripts_home/14_graph_rag_answer.py", query,
            "--graph", TOPIC_GRAPH, "--top-seeds", top_seeds, "--max-topics", max_topics,
        )
    if n == 4:
        return _run_capture(
            "scripts_home/15_search_agent_demo.py", query,
            "--topics", TOPICS, "--pages", RAW_PAGES,
            "--top-k", top_k_agent, "--max-steps", max_steps,
        )
    # 새 파이프라인을 추가하려면 위와 같은 패턴으로 분기를 늘리고
    # PIPELINE_TITLES / compare(which=...) 기본값만 맞춰 주면 됩니다.
    raise ValueError(n)


def _render(query, results, show_logs):
    try:
        from IPython.display import Markdown, display
        use_md = True
    except Exception:
        use_md = False

    if use_md:
        md = [f"### 질문\n\n> {query}\n"]
        for n, log, answer, elapsed, code in results:
            status = "✅" if code == 0 else "⚠️"
            md.append(f"### {status} {PIPELINE_TITLES[n]}  · _{elapsed:.1f}s_\n\n{answer or '(답변 없음)'}\n")
            if show_logs and log:
                md.append("<details><summary>retrieval log 보기</summary>\n\n```\n" + log[:6000] + "\n```\n</details>\n")
            md.append("\n---\n")
        display(Markdown("\n".join(md)))
    else:
        print("=" * 90)
        print("질문:", query)
        for n, log, answer, elapsed, code in results:
            print("\n" + "=" * 90)
            print(f"{PIPELINE_TITLES[n]}  ({elapsed:.1f}s, exit={code})")
            print("-" * 90)
            if show_logs and log:
                print(log[:4000]); print("-" * 90)
            print(answer or "(답변 없음)")


def compare(query, *, top_k_raw=10, top_k_topic=5, top_seeds=8, max_topics=30,
            top_k_agent=4, max_steps=4, show_logs=False, which=(1, 2, 3, 4)):
    """같은 질문을 네 파이프라인에 넣고 [(n, log, answer, elapsed, code), ...]를 반환."""
    results = []
    for n in which:
        print(f"\u25b6 {PIPELINE_TITLES[n]} 실행 중...", flush=True)
        try:
            code, out, elapsed = _run_pipeline(n, query, top_k_raw, top_k_topic, top_seeds, max_topics, top_k_agent, max_steps)
            log, answer = _split_answer(out)
            results.append((n, log, answer, elapsed, code))
        except Exception as e:
            results.append((n, "", f"[실행 실패] {e!r}", 0.0, -1))
    _render(query, results, show_logs)
    return results


print("compare() 준비 완료")

compare() 준비 완료


### 17-b. Retrieval 성능 지표 (label-free)

정답 라벨 없이 `compare(...)` 결과만으로 계산하는 지표 3가지입니다.

- **score 분포 / 확신도** : top-1 score가 top-k 평균보다 얼마나 높은지(margin). margin이 클수록 검색이 한 문서에 "확신"을 가졌다는 뜻
- **문서 겹침도 (Jaccard)** : 파이프라인들이 찾아온 문서(파일) 집합이 서로 얼마나 겹치는지. 겹침이 낮으면 같은 질문에도 서로 다른 근거를 봤다는 뜻
- **Latency** : 파이프라인별 실행 시간(`compare()`가 측정한 `elapsed`)

retrieval log의 각 줄에서 `file=`과 `score=`를 함께 뽑아 `(score, file_id)`로 파싱하므로, 청크/토픽/그래프 어느 로그 형식이든 동작합니다.

In [30]:
import re
from statistics import mean

# retrieval log 한 줄에서 file=, score=를 함께 찾으면 (score, file_id)로 본다.
# (청크/토픽/그래프 로그의 chunk= / topic= / outcome= 등 형식 차이에 영향받지 않음)
_SCORE_RE = re.compile(r"score=([\d.]+)")
_FILE_RE = re.compile(r"file=(\S+?)(?:\.pdf)?(?=[\s\]]|$)")


def _parse_retrieval_log(n, log):
    pairs = []
    if not log:
        return pairs
    for line in log.splitlines():
        sm = _SCORE_RE.search(line)
        fm = _FILE_RE.search(line)
        if sm and fm:
            pairs.append((float(sm.group(1)), fm.group(1)))
    return pairs


def _score_stats(pairs, top_k=5):
    if not pairs:
        return {"top1": None, "topk_mean": None, "margin": None, "n": 0}
    scores = [s for s, _ in pairs[:top_k]]
    top1 = scores[0]
    topk_mean = mean(scores)
    return {"top1": round(top1, 3), "topk_mean": round(topk_mean, 3),
            "margin": round(top1 - topk_mean, 3), "n": len(pairs)}


def _jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return None
    return len(set_a & set_b) / len(union)


def _metrics_to_markdown(parsed, top_k=5):
    lines = [f"| 파이프라인 | top-1 score | top-{top_k} 평균 | margin | 검색 건수 | latency(s) |",
             "|---|---|---|---|---|---|"]
    for n in sorted(parsed):
        s = parsed[n]["score_stats"]
        lines.append(f"| P{n} | {s['top1']} | {s['topk_mean']} | {s['margin']} | {s['n']} | {parsed[n]['elapsed']:.1f} |")
    score_table = "\n".join(lines)

    overlap_lines = ["| 비교 | 겹친 문서 수 | Jaccard |", "|---|---|---|"]
    ns = sorted(parsed)
    for i in range(len(ns)):
        for j in range(i + 1, len(ns)):
            a, b = ns[i], ns[j]
            jac = _jaccard(parsed[a]["case_ids"], parsed[b]["case_ids"])
            inter = len(parsed[a]["case_ids"] & parsed[b]["case_ids"])
            jac_str = f"{jac:.2f}" if jac is not None else "n/a"
            overlap_lines.append(f"| P{a} \u2229 P{b} | {inter} | {jac_str} |")
    return score_table, "\n".join(overlap_lines)


def compute_metrics(results, top_k=5, show_log_excerpt=False):
    """compare()가 반환한 results로 score 분포·확신도·Jaccard·latency를 계산하고 dict를 반환."""
    parsed = {}
    for n, log, answer, elapsed, code in results:
        pairs = _parse_retrieval_log(n, log)
        parsed[n] = {
            "pairs": pairs,
            "case_ids": {cid for _, cid in pairs},
            "elapsed": elapsed,
            "score_stats": _score_stats(pairs, top_k=top_k),
        }
        if show_log_excerpt:
            print(f"-- P{n} retrieval log 일부 (파싱된 {len(pairs)}건) --")
            print((log or "")[:400]); print()

    score_table, overlap_table = _metrics_to_markdown(parsed, top_k=top_k)
    try:
        from IPython.display import Markdown, display
        display(Markdown("**Score 분포 / 확신도**\n\n" + score_table))
        display(Markdown("**문서 겹침도 (Jaccard)**\n\n" + overlap_table))
    except Exception:
        print("Score 분포 / 확신도\n" + score_table + "\n")
        print("문서 겹침도 (Jaccard)\n" + overlap_table)
    return parsed


print("compute_metrics() 준비 완료")

compute_metrics() 준비 완료


### 17-c. 비교용 질문 3종

`three_rag_pipeline_comparison.ipynb`에서 쓰던 세 질문을 그대로 가져왔습니다.
각 질문은 한 파이프라인이 상대적으로 강한 유형을 노립니다.

- `Q1` — 파이프라인 1(Chunk BM25) 지향: 수치·조건이 규정 원문에 그대로 나오는 키워드형
- `Q2` — 파이프라인 2(Topic-based) 지향: 법률 용어 대신 개인 상황을 풀어 묻는 상담형
- `Q3` — 파이프라인 3(Graph RAG) 지향: 지역·주택유형별 규제 패턴을 비교하는 글로벌형

`compare(...)` 안의 질문 문자열만 바꾸면 얼마든지 다시 테스트할 수 있습니다.

In [31]:
Q1 = "서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로 청약하려면 청약통장 예치금이 정확히 얼마 있어야 해?"
Q2 = "본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주민등록표에 모시고 있습니다. 그런데 아버지가 본인 명의의 아파트를 한 채 소유하고 계십니다. 가점제 청약 시 부모님 두 분 다 제 부양가족으로 점수를 받을 수 있나요?"
Q3 = "현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이 원천적으로 차단되는 지역이나 주택 유형은 무엇이며, 반대로 다주택자도 1순위 당첨을 노려볼 수 있는 규제 패턴(지역 및 추첨제 비율)은 어떻게 연결되어 있나요?"

print("Q1:", Q1[:40], "...")
print("Q2:", Q2[:40], "...")
print("Q3:", Q3[:40], "...")

Q1: 서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로  ...
Q2: 본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주 ...
Q3: 현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이  ...


### 17-1. 파이프라인 1 지향 질문 (Chunk BM25가 비교적 잘 잡는 유형)

In [32]:
results1 = compare(Q1, show_logs=True)

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Topic-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...
▶ 파이프라인 4 · Search Agent (multi-step) 실행 중...


### 질문

> 서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로 청약하려면 청약통장 예치금이 정확히 얼마 있어야 해?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _13.9s_

서울 거주자가 **전용면적 102㎡ 이하 민영주택**에 청약하려면, **청약통장 예치금이 600만원**이어야 합니다. 서울은 **특별시**에 해당하므로, 102㎡ 이하의 예치기준금액은 **600만원**입니다.[2]

다만, 민영주택 청약은 **입주자모집공고일 기준**으로 해당 면적의 예치기준금액이 충족되어 있어야 하고, 청약통장 종류와 1순위 자격도 함께 충족해야 합니다.[2][4]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=21.083 file=청약홈 FAQ - 02.청약통장 chunk=4
   변경] Q. 경기도 거주자로서 수도권 거주 자격으로 서울에 공급되는 주택에 청약하고 싶습니다. 청약예금을 서울 지역 예치금으로 증액해야 하나요? A. 아닙니다. 수도권 거주자 자격으로 청약신청하는 경우 입주자모집공고일 기준 본인이 거주하는 지역(그 밖의 광역시)의 예치금액을 충족하고 신청가능 순위에 해당하면 가능합니다. * 청약예금 및 부금 가입자의 경우 금액 및 면적 변경은 입주자모집공고일
2. score=18.834 file=주택청약 FAQ.pdf chunk=74
    주택 구입 시 : ’22년 말까지 우대이율 적용 31 제2순위로 당첨된 경우 해당 통장은 재사용이 가능한지?    제2순위 청약신청 또한 입주자저축을 필요로 하며, 입주자저축 통장을 사용하여 분양주택  또는 분양전환공공임대주택의 입주자로 선정된 경우에는 해당 통장은 재사용할 수 없습 니다. 32 주민등록 거주지가 인천시인 청약신청자가 서울에서 분양하는 전용면적 85㎡이하의  민영주택을 청약
3. score=18.738 file=주택청약 FAQ.pdf chunk=4
   ······22 Q32. 주민등록 거주지가 인천시인 청약신청자가 서울에서 분양하는 전용면적 85㎡이하의  민영주택을 청약코자 할 때 예치기준금액은? ···························································22 Q33. 인천광역시 거주자로 청약예금 400만원(전용면적 102㎡ 이하)에 가입한 자가  입주자모집공고일 전 서울시로 이주한 경우 10
4. score=16.969 file=청약홈 FAQ - 02.청약통장 chunk=0
   [page 1] [청약통장의 가입] Q. 청약통장이란? 청약통장의 종류는? A. 청약통장이란? - 청약통장(입주자저축)은 주택법에 따라 주택을 공급받으려는 자에게 입주금의 일부나 전부를 저축하도록 하는 제도입니다. - 특별공급(기관추천 특별공급 중 국가유공자, 장애인, 철거민 등은 청약통장 가입 불필요)·일반공급1·2순위로 청약을 신청하기 위해서는 청약통장을 보유하고 있어야 합니다. - 청약
5. score=14.243 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=0
   [page 1] [ApplyHome접속] Q. 구글 크롬 브라우저로도 청약할 수 있나요? A. Microsoft Edge, 구글 크롬, 모질라 파이어폭스 브라우저 등에서 청약신청 하실 수 있으며 다른 메뉴도 이용할 수 있습니다.  [page 2] [ApplyHome접속] Q. 보안프로그램을 꼭 설치해야 하나요? A. 청약Home에서는 청약신청 및 기타 여러 메뉴에서 공동인증서 사용과 개인정보
6. score=14.106 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=1
    [page 5] [ApplyHome접속] Q. 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하나요? A. 반드시 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하는 것은 아닙니다. 다른 은행이라도 상관없으나, ①공동인증서(은행보험용 또는 범용) 및 ②금융인증서, ③네이버인증서, ④KB국민인증서, ⑤토스인증서, ⑥신한인증서가 발급되어 있어야 청약 신청이 가능합니다.  [page 6] [
7. score=13.665 file=주택청약 FAQ.pdf chunk=75
        청약부금 가입자가 전용면적 85㎡초과 민영주택에 청약하기 위해서는 입주자모집공고일  전일까지 거주지역별 민영주택 청약 예치기준금액을 충족하는 청약예금으로 전환해야  변경 후 면적에 청약신청이 가능합니다.      청약저축을 청약예금으로 전환하여 민영주택에 청약하고자 하는 경우에는 반드시 입주자 모집공고일 전일까지 은행영업점에서 전환을 완료하여야 합니다.  [page 56] 24 35
8. score=13.504 file=청약홈 FAQ - 02.청약통장 chunk=3
   도로 사용하려 하는데 가능한가요? A. 차액인출이 가능합니다. 자세한 내용은 청약통장 가입은행으로 문의해주시기 바랍니다.  [page 10] [청약통장의 변경] Q. 청약예금이나 청약부금의 면적 또는 예치금액 변경은 언제까지 해야 하나요? A. 청약하려는 주택의 입주자모집공고일 전일까지 면적 또는 상당하는 예치금으로 변경해야 합니다. * 주택청약종합저축 가입자는 청약신청 당일까지 주택규모를 
9. score=13.084 file=청약 자격 체크리스트 (applyhome_checklist) chunk=2
    및 공시가격 1억원(수도권 1억6천만원) 이하 주택 또는 분양권 등 / 나. 전용면적 85㎡ 이하 및 공시가격 3억원(수도권 5억원) 이하 주택 또는 분양권 등   - cases_and_notes: 공공임대주택 청약 시에는 미적용. * 제10호 / 선착순 공급 분양권등 소유   - description: 입주자 선정 후 미계약분을 선착순 방법으로 공급받아 분양권등을 소유하고 있는 경우   
10. score=10.128 file=청약홈 FAQ - 02.청약통장 chunk=2
   별 청약예금 예치금액 이상인 계좌의 경우 해당 주택규모의 청약예금으로 전환할 수 있습니다. * 납입인정금액은 순위확인서 발급을 통해 직접 확인할 수 있습니다. (청약통장(입주자저축) 가입은행 방문 발급 또는 청약홈 > 청약자격확인 > 청약통장 > 순위확인서 발급 메뉴 이용) 청약저축을 청약예금으로 전환한 경우, 민영주택 청약은 바로 가능합니다. 다만, 청약하고자 하는 주택의 최초 입주자모집공
```
</details>


---

### ✅ 파이프라인 2 · Topic-based (structural)  · _22.9s_

제공된 근거만으로는 **서울 거주자, 전용면적 102㎡ 이하 민영주택의 1순위 청약에 필요한 예치금 “정확한 금액”**을 확인할 수 없습니다. 근거에는 **민영주택 청약에는 청약예금·청약부금(또는 주택청약종합저축의 민영주택 청약 가능 조건)이 필요하고**, 청약저축을 민영주택용으로 전환하려면 **지역별 청약예금 예치금액 이상**이어야 한다는 점만 나와 있습니다.[R1][R2]

즉, 이 질문에 답하려면 **서울 지역의 전용면적 102㎡ 이하에 해당하는 지역별 예치금액표**가 추가로 필요합니다. 현재 제공된 자료에는 그 표가 없어 정확한 금액을 단정할 수 없습니다.[R2]

정리하면:
- **1순위 민영주택 청약 자체는 청약통장 보유가 전제**됩니다.[R1]
- **청약저축의 경우** 민영주택 청약을 하려면 **해당 지역·면적의 청약예금 예치금 이상**이어야 합니다.[R2]
- 하지만 **서울 / 102㎡ 이하의 정확한 예치금 숫자**는 이번 제공 자료에 없습니다.[R2]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Topics
1. score=10.228 file=청약홈 FAQ - 02.청약통장 topic=0
   질문: 청약통장이란 무엇이고, 어떤 종류가 있나요?
   결과: 필요한 청약통장 종류와 가입 필요 여부를 이해하고 청약 준비가 가능하다.
2. score=8.100 file=주택공급에 관한 규칙.pdf topic=1
   질문: 청약하려면 주택청약종합저축이 꼭 있어야 하나요?
   결과: 가입요건을 충족하면 청약 신청이 가능하고, 이미 사용한 통장은 재사용이 제한된다.
3. score=5.945 file=주택청약 FAQ.pdf topic=4
   질문: 주택을 소유했다고 보지 않는 예외에 해당하면 무주택으로 인정되나요?
   결과: 요건을 충족하면 해당 주택을 소유하지 않은 것으로 보아 무주택 자격을 인정받을 수 있다.
4. score=4.919 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=4
   질문: 청약통장 가입은행 인터넷뱅킹에 가입되어 있어야 하나요?
   결과: 인증서가 있으면 가입은행 인터넷뱅킹이 아니어도 청약신청이 가능하다.
5. score=4.229 file=청약홈 FAQ - 03.청약가점제 topic=1
   질문: 청약가점제 적용비율은 면적과 지역에 따라 어떻게 달라지나요?
   결과: 해당 주택 유형에서 가점제와 추첨제의 적용 비율을 확인할 수 있다.

## Added Raw Chunks
R1. score=0.000 file=청약홈 FAQ - 02.청약통장 chunk=0
    [page 1] [청약통장의 가입] Q. 청약통장이란? 청약통장의 종류는? A. 청약통장이란? - 청약통장(입주자저축)은 주택법에 따라 주택을 공급받으려는 자에게 입주금의 일부나 전부를 저축하도록 하는 제도입니다. - 특별공급(기관추천 특별공급 중 국가유공자, 장애인, 철거민 등은 청약통장 가입 불필요)·일반공급1·2순위
R2. score=0.000 file=청약홈 FAQ - 02.청약통장 chunk=1
    입] Q. 과거 아파트 당첨사실이 있는데 청약통장에 신규 가입하면 1순위가 될 수 있나요? A. 네. 1순위 청약신청 자격이 되면 가능합니다. * 청약통장을 사용하여 분양주택 또는 분양전환공공임대주택의 입주자로 선정된 경우, 동일한 통장으로 다른 주택에 청약할 수 없습니다. 과거 당첨된 청약통장을 해지한 후 신규 가입하셔
R3. score=0.000 file=주택공급에 관한 규칙.pdf chunk=0
    [page 1] 주택공급에 관한 규칙 [시행 2026. 6. 15.] [국토교통부령 제1592호, 2026. 6. 15., 일부개정]          제1장 총칙              제1조(목적)              제2조(정의)              제3조(적용대상)              제4조(주택의 공급대
R4. score=0.000 file=주택공급에 관한 규칙.pdf chunk=1
            제24조(주택공급 신청 서류의 관리)          제4절 주택 사전청약              제24조의2(사전당첨자 모집 시기)              제24조의3(사전당첨자 모집 승인 및 통보)              제24조의4(사전당첨자모집공고)              제24조의5(주택공급신청서 
R5. score=0.000 file=주택청약 FAQ.pdf chunk=0
    [page 1] [추가 정정]  [page 2] 본 FAQ는 2022년 7월 기준으로 작성되었으며, 관련 법령의 개정, 법령해석  변경 등에 따라 변경될 수 있음을 알려드리며, 참고용으로만 활용하여 주시기  바랍니다.   또한 개별 사실관계의 변동 등으로 인한 유사사례인 경우에 다른 해석이 있을  수 있으며, 본 FAQ를
R6. score=0.000 file=주택청약 FAQ.pdf chunk=1
    ························8 Q6. 「주택공급에 관한 규칙」 제34조에 따른 대규모택지개발지구에서 주택이 공급되는 경우  일반공급 뿐만 아니라 특별공급 물량도 공급비율에 따라 배정되는지? ···························8 Q7. 행정중심복합도시예정지역에서 공급하는 주택의 경우 공급비율 
R7. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=0
    [page 1] [ApplyHome접속] Q. 구글 크롬 브라우저로도 청약할 수 있나요? A. Microsoft Edge, 구글 크롬, 모질라 파이어폭스 브라우저 등에서 청약신청 하실 수 있으며 다른 메뉴도 이용할 수 있습니다.  [page 2] [ApplyHome접속] Q. 보안프로그램을 꼭 설치해야 하나요? A. 청약
R8. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=1
     [page 5] [ApplyHome접속] Q. 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하나요? A. 반드시 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하는 것은 아닙니다. 다른 은행이라도 상관없으나, ①공동인증서(은행보험용 또는 범용) 및 ②금융인증서, ③네이버인증서, ④KB국민인증서, ⑤토스인증서, ⑥신
R9. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=0
    [page 1] [청약가점제 내용] Q. 청약가점제란? A. 가점항목에 대하여 산정한 점수(가점제 점수)가 높은 순으로 입주자를 선정하는 방식을 말합니다. 민영주택 일반공급 입주자 선정 시 일정비율(나머지 비율은 추첨으로 선정)을 가점제로 선정합니다. □ 가점항목 항 목 점 수 범 위 무주택기간 32점 1년미만~15년이상
R10. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=1
    주민등록표등본에 등재된 직계존속) 단, 직계존속 및 그 배우자 중 한 명이라도 주택을 소유(분양권 등 포함)하고 있는 경우 부양가족으로 보지 않으나 공급규칙 제53조에 해당하는 주택을 가지고 있는 경우 아래와 같이 판단 *주택공급에 관한 규칙 제53조(주택소유여부 판정기준) 적용 사례 - 직계존속 또는 그 배우자가 소유하
```
</details>


---

### ⚠️ 파이프라인 3 · Graph RAG (relational)  · _10.3s_

## Seed Phrase Nodes
1. score=15.52 type=Screen label=청약통장
2. score=10.90 type=Setting label=전용면적 85㎡ 이하 등 면적기준
3. score=8.00 type=Setting label=민영주택
4. score=6.04 type=Task label=청약통장 종류 전환
5. score=6.04 type=Task label=청약통장 가입내역 확인
6. score=6.04 type=Setting label=본인 청약통장 가입기간
7. score=6.04 type=Setting label=배우자 청약통장 가입기간
8. score=5.56 type=Task label=거주지와 청약통장 가입지역 확인

## Graph Retrieval Summary
expanded_phrase_nodes=8
retrieved_topics=6
analysis_mode=overview
accepted_topics=0
rejected_topics=0
other_topics=6

## Representative Topics
Top graph matches:
  [C1] file=주택청약 FAQ.pdf topic=2 outcome=청약통장의 전환·수정·관리 가능 범위를 확인하여 청약자격 유지 여부를 판단할 수 있다 score=21.55
       작업/질문: 청약통장을 전환하거나 예치금·납입내역을 변경할 수 있나요?
       결론: 청약통장의 전환·수정·관리 가능 범위를 확인하여 청약자격 유지 여부를 판단할 수 있다.
  [C2] file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=10 outcome=민영주택 청약가점을 확인할 수 있다 score=12.07
       작업/질문: 민영주택 가점제는 어떤 항목으로 계산되나요?
       결론: 가점 항목을 바탕으로 민영주택 청약가점을 확인할 수 있다.
  [C3] file=주택청약 FAQ.pdf topic=4 outcome=요건을 충족하면 해당 주택을 소유하지 않은 것으로 보아 무주택 자격을 인정받을 수 있다 score=10.90
       작업/질문: 주택을 소유했다고 보지 않는 예외에 해당하면 무주택으로 인정되나요?
       결론: 요건을 충족하면 해당 주택을 소유하지 않은 것으로 보아 무주택 자격을 인정받을 수 있다.
  [C4] file=주택공급에 관한 규칙.pdf topic=0 outcome=청약 신청 자격이 인정됨 score=8.00
       작업/질문: 내가 이 지역 청약 대상자가 될 수 있는지 어떻게 판단하나요?
       결론: 공급대상에 해당하면 청약 신청 자격이 인정된다.
  [C5] file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=6 outcome=가입지역을 맞추면 민영주택 청약순위 조회가 정상화된다 score=6.04
       작업/질문: 민영주택 청약순위가 생각과 다르게 보이는 이유는 무엇인가요?
       결론: 가입지역을 맞추면 민영주택 청약순위 조회가 정상화된다.


[stderr]
Traceback (most recent call last):
  File "D:\snu_python\graphrag\scripts_home\14_graph_rag_answer.py", line 477, in <module>
    main()
  File "D:\snu_python\graphrag\scripts_home\14_graph_rag_answer.py", line 463, in main
    answer = answer_with_llm(
             ^^^^^^^^^^^^^^^^
  File "D:\snu_python\graphrag\scripts_home\14_graph_rag_answer.py", line 337, in answer_with_llm
    response = client.chat.completions.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_utils\_utils.py", line 287, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\resources\chat\completions\completions.py", line 1147, in create
    return self._post(
           ^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_base_client.py", line 1259, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_base_client.py", line 1047, in request
    raise self._make_status_error_from_response(err.response) from None
openai.BadRequestError: Error code: 400 - {'error': {'message': 'Selected model is at capacity. Please try a different model.'}}


---

### ✅ 파이프라인 4 · Search Agent (multi-step)  · _34.4s_

## User Query
서울에 거주하는 사람이 전용면적 102제곱미터 이하 민영주택에 1순위로 청약하려면 청약통장 예치금이 정확히 얼마 있어야 해?

## Step 1: search_raw
reason: 서울 거주자가 전용면적 102제곱미터 이하 민영주택에 1순위로 청약할 때 필요한 정확한 예치금 기준 문구를 원문 규정에서 확인하기 위해
query: 서울 민영주택 전용면적 102㎡ 이하 청약통장 예치금 1순위 예치금 기준
S1-1. raw score=26.852 file=청약홈 FAQ - 02.청약통장 chunk=2
   별 청약예금 예치금액 이상인 계좌의 경우 해당 주택규모의 청약예금으로 전환할 수 있습니다. * 납입인정금액은 순위확인서 발급을 통해 직접 확인할 수 있습니다. (청약통장(입주자저축) 가입은행 방문 발급 또는 청약홈 > 청약자격확인 > 청약통장 > 순위확인서 발급 메뉴 이용) 청약저축을 청약예금으로 전환한 경우, 민영주택 청약은 바로 가능합니다. 다만, 청약하고자 하는 주택의 최초 입주자모집공
S1-2. raw score=21.968 file=청약 자격 체크리스트 (applyhome_checklist) chunk=5
   등재 후 공고일 현재 세대주로 변경했어도 3년 연속 등재 시 인정. 가족관계증명서 기준이므로 생모여도 미등재 시 불인정.   - descendants:     - criteria: 미혼 자녀 및 부모가 모두 사망한 미혼 손자녀.     - rules_by_age:       - under_30: 공고일 현재 계속하여 90일을 초과하여 해외 체류 중인 경우 부양가족 제외       - over
S1-3. raw score=21.326 file=주택청약 FAQ.pdf chunk=74
   주택 구입 시 : ’22년 말까지 우대이율 적용 31 제2순위로 당첨된 경우 해당 통장은 재사용이 가능한지?    제2순위 청약신청 또한 입주자저축을 필요로 하며, 입주자저축 통장을 사용하여 분양주택  또는 분양전환공공임대주택의 입주자로 선정된 경우에는 해당 통장은 재사용할 수 없습 니다. 32 주민등록 거주지가 인천시인 청약신청자가 서울에서 분양하는 전용면적 85㎡이하의  민영주택을 청약코
S1-4. raw score=20.297 file=주택청약 FAQ.pdf chunk=145
   민영주택 생애최초 특별공급 신청이 가능한지?    기본적으로 민영주택은 청약부금, 청약예금, 주택청약종합저축 통장으로만 청약신청이  가능하나, 청약저축 가입자의 경우에는 입주자 모집공고일 전날까지 청약저축 통장을  청약예금 통장으로 전환하여야 인정이 가능합니다. * 청약저축 통장은 국민주택 청약 시에만 이용 가능하며, 청약저축을 청약예금으로 전환한 후에  청약저축으로 재전환은 불가능  [pa

## Step 2: finish
reason: 이미 검색 결과에 서울(특별시) 거주자의 전용면적 102㎡ 이하 민영주택 예치기준금액이 확인되어, 추가 검색 없이 답변 단계로 넘어갈 수 있습니다.

## Stop Reason
planner selected finish at step 2

## Final Answer
서울에 거주하는 경우, 전용면적 102㎡ 이하 민영주택에 1순위로 청약하려면 청약예금 예치기준금액이 **600만원**이어야 합니다. 서울은 **특별시**로 분류되며, 102㎡ 이하 기준이 특별시·부산광역시 **600만원**으로 제시되어 있습니다. [S1-3]

또한 예치기준금액은 **청약신청자의 거주지 기준**으로 판단하므로, 입주자모집공고일 현재 서울 거주자인지 확인해야 합니다. [S1-2][S1-3]

정리하면, **서울 거주자 + 전용면적 102㎡ 이하 민영주택 + 1순위 청약 = 예치금 600만원**입니다. [S1-3]


---


### 17-2. 파이프라인 2 지향 질문 (topic 구조화가 유리한 유형)

In [33]:
results2 = compare(Q2, show_logs=True)

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Topic-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...
▶ 파이프라인 4 · Search Agent (multi-step) 실행 중...


### 질문

> 본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주민등록표에 모시고 있습니다. 그런데 아버지가 본인 명의의 아파트를 한 채 소유하고 계십니다. 가점제 청약 시 부모님 두 분 다 제 부양가족으로 점수를 받을 수 있나요?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _13.4s_

아니요. 가점제 청약에서 직계존속은 신청자가 세대주이고, 입주자모집공고일 기준 최근 3년 이상 계속하여 같은 세대별 주민등록표에 등재되어 있어야 부양가족으로 인정됩니다. 다만 직계존속과 그 배우자 중 한 명이라도 주택을 소유하고 있으면, 직계존속과 그 배우자 모두 부양가족으로 보지 않습니다.[3][6]

질문 사례에서는 아버지께서 본인 명의의 아파트를 1채 소유하고 있으므로, 아버지는 물론 어머니도 함께 부양가족으로 인정되지 않습니다.[1][3][7]

즉, 가점제 청약 시 부모님 두 분 다 부양가족 점수를 받을 수 없습니다.[1][3]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=51.285 file=청약홈 FAQ - 03.청약가점제 chunk=3
    1월부터 다시 본인의 세대에 합가했는데, 이 경우 부모님은 부양가족으로 인정되나요? A. 직계존속은 신청자와 동일한 주민등록표등본에 3년 이상 ‘연속해서’ 등재되어 있어야하므로 합가한 2019년 1월부터 3년 이상이 경과 해야 비로소 부양가족으로 인정됩니다. 다만, 직계존속과 직계존속의 배우자 중 한 명이라도 주택을 소유(분양권등 포함)하고 있는 경우에는 직계존속과 그 배우자 모두 부양가족
2. score=27.463 file=청약홈 FAQ - 03.청약가점제 chunk=4
   본인과 주민등록표등본상 분리되어 있을 경우 부양가족수에 포함되나요? A. 포함되지 않습니다. 자녀 (부모가 모두 사망한 손자녀 포함)의 경우 입주자모집공고일 현재 주택공급신청자 또는 그 배우자와 같은 세대별 주민등록표에 등재된 미혼인 자녀에 한정하여 부양가족으로 인정합니다. 또한, 주택공급신청자의 만30세 이상인 직계비속 은 입주자모집공고일을 기준으로 최근 1년 이상 계속하여 주택공급신청자 
3. score=26.925 file=주택청약 FAQ.pdf chunk=123
   등으로  법적 부부의 관계가 확인이 되면 주민등록표등본에 등재되지 않더라도 부양가족으로  인정을 받을 수 있습니다. ■ 주택공급에 관한 규칙 [별표 1] 1. 가점제 적용기준  나. 부양가족의 인정 적용기준   1) 부양가족은 입주자모집공고일 현재 주택공급신청자 또는 그 배우자(주택공급신청자와 같은  세대별 주민등록표에 등재되어 있지 않은 배우자를 포함한다. 이하 이 목에서 같다)와 같은  
4. score=26.565 file=주택청약 FAQ.pdf chunk=122
   가 만30세가 된 날 또는 신청자 및 배우자가  무주택자가 된 날 중 늦은 날 ~ 입주자 모집공고일 O 혼인신고일 또는 청약신청자 및 배우자가 무주택자가 된 날 중  늦은 날 ~ 입주자 모집공고일  [page 115] Ⅱ. 일반공급 83 151 1주택을 소유하고 있는 경우 민영주택 청약을 위한 가점제 항목 중 무주택기간 점수는  몇점으로 산정하여야 하는지?    수도권에 지정된 공공주택지구,
5. score=26.541 file=주택청약 FAQ.pdf chunk=78
   자인지?   청약신청자와 주택을 소유한 전혼자녀의 배우자가 동일한 세대별 주민등록표에 등재 되어 있는 경우 세대원으로 인정되지 않으므로, 청약신청자의 주택소유 판단 시 영향을  미치지 않습니다.  [page 60] 28 43 무주택자인 아내가 유주택자인 남편과 주민등록표상 분리되어 친정부모의 세대별  주민등록표에 등재되어 있는 경우, 무주택자인 친정부모는 무주택세대구성원 자격이  인정되는지?
6. score=24.955 file=주택청약 FAQ.pdf chunk=344
   은 배우자를 포함한다. 이하 이 목에서  같다)와 같은 세대별 주민등록표에 등재된 세대원으로 한다. 다만, 자녀의 경우 미혼 으로 한정한다.  [page 353] 주택공급에 관한 규칙(법령) 321   2) 주택공급신청자 또는 그 배우자의 직계존속은 주택공급신청자가 입주자모집공고일 현 재 세대주인 경우로서 입주자모집공고일을 기준으로 최근 3년 이상 계속하여 주택공 급신청자 또는 그 배우자와 
7. score=23.871 file=주택청약 FAQ.pdf chunk=162
   은 영향을 미치지 않으며 다만, 가점제 부양가족 산정 시에는 피부양자 여부와 무관하게 직계존속의 배우자가  주택을 소유하고 있다면 그 직계존속은 부양가족에서 제외하여야 합니다. * 「주택공급에 관한 규칙」 [별표1] 제1호나목2) 주택공급신청자 또는 그 배우자의 직계존속은  주택공급신청자가 입주자모집공고일 현재 세대주인 경우로서 입주자모집공고일을 기준으로  최근 3년 이상 계속하여 주택공급신
8. score=23.843 file=주택청약 FAQ.pdf chunk=126
   에서 청약신청자 또는 배우자와 단 하루라도 주민등록이 분 리된 이력이 있다면 다시 전입된 그 날부터 부양기간을 재산정하여야 합니다. 162 증빙서류로 제출한 가족관계증명부에는 생모가 등재되어 있지 않으나 주민등록 초본에는 생모가 등재되어 있는 경우 직계존속으로 인정받을 수 있는지?    「주택공급에 관한 규칙」 제23조제2항제2호에 따르면 주택청약 신청자는 세대주 또는  세대원인 사실 등을 
9. score=23.498 file=청약홈 FAQ - 03.청약가점제 chunk=7
   년인 세대주로, 모친은 만68세이며 3채의 아파트로 임대사업을 하고 있습니다. 본인은 무주택자에 해당하나요? A. 분양주택에 가점제 청약 시 무주택자에 해당합니다. 만60세 이상인 직계존속이 주택을 소유한 경우 청약신청자를 무주택자로 간주하므로 모친이 여러채의 주택을 소유하더라도 신청자 본인은 무주택자입니다. * 단, 노부모부양 특별공급 또는 공공사업시행자(국가, 지자체, LH 및 지방공사 
10. score=22.367 file=주택청약 FAQ.pdf chunk=161
   경우에는 계속하여 같은 세대별 주민등록표상에 등재된 것으로 볼 수 없으므로 노부모  부양 특별공급 신청 자격이 제한되며, 이에 대한 확인을 위해 증빙서류 제출 시 출입국 사실증명원을 제출하여 해외 체류여부를 증명하여야 합니다. 256 노부모 특별공급 청약 시 청약신청자의 어머니는 청약자와 동일한 주민등록에  등재되어 있으나 아버지는 주민등록이 분리되어 있는 경우, 아버지도 무주택이어야  하는
```
</details>


---

### ✅ 파이프라인 2 · Topic-based (structural)  · _22.0s_

아니요. 현재 정보만 보면 **두 분 모두 부양가족으로 인정받기 어렵습니다**. 청약가점제에서는 직계존속을 부양가족으로 보려면 **신청자가 세대주이고, 3년 이상 같은 주민등록표등본에 등재**되어 있어야 하며, **직계존속과 그 배우자 중 한 명이라도 주택을 소유하면 부양가족으로 보지 않습니다**. 질문의 경우 아버지가 본인 명의 아파트를 소유하고 있으므로, 어머니가 무주택자이더라도 **부모님 두 분 모두 부양가족 수에 포함되지 않는 것으로 봐야 합니다**. [R1][R2]

다만 **그 아파트가 주택공급에 관한 규칙 제53조의 예외에 해당하는 주택**이라면 판단이 달라질 수 있습니다. 그런 예외에 해당하는지 여부는 제공된 정보만으로는 확인되지 않습니다. [R1][R2]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Topics
1. score=14.519 file=청약홈 FAQ - 03.청약가점제 topic=2
   질문: 청약가점제에서 부양가족수는 누구를 어떻게 인정하나요?
   결과: 인정되는 부양가족 수에 따라 가점이 산정된다.
2. score=8.332 file=청약홈 FAQ - 03.청약가점제 topic=0
   질문: 청약가점제란 무엇이며 당첨자는 어떻게 선정되나요?
   결과: 가점제 점수에 따라 가점제 배정 물량에서 당첨 여부가 결정된다.
3. score=7.530 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=10
   질문: 민영주택 가점제는 어떤 항목으로 계산되나요?
   결과: 가점 항목을 바탕으로 민영주택 청약가점을 확인할 수 있다.
4. score=6.407 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=21
   질문: 순위확인서는 무엇이며 언제 필요한가요?
   결과: 필요한 경우 순위확인서를 발급해 청약자격을 증명할 수 있다.
5. score=6.291 file=청약홈 FAQ - 03.청약가점제 topic=3
   질문: 청약가점제에서 무주택기간과 무주택자 여부는 어떻게 판단하나요?
   결과: 무주택자 해당 여부와 무주택기간이 산정되어 가점제 점수에 반영된다.

## Added Raw Chunks
R1. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=0
    [page 1] [청약가점제 내용] Q. 청약가점제란? A. 가점항목에 대하여 산정한 점수(가점제 점수)가 높은 순으로 입주자를 선정하는 방식을 말합니다. 민영주택 일반공급 입주자 선정 시 일정비율(나머지 비율은 추첨으로 선정)을 가점제로 선정합니다. □ 가점항목 항 목 점 수 범 위 무주택기간 32점 1년미만~15년이상
R2. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=1
    주민등록표등본에 등재된 직계존속) 단, 직계존속 및 그 배우자 중 한 명이라도 주택을 소유(분양권 등 포함)하고 있는 경우 부양가족으로 보지 않으나 공급규칙 제53조에 해당하는 주택을 가지고 있는 경우 아래와 같이 판단 *주택공급에 관한 규칙 제53조(주택소유여부 판정기준) 적용 사례 - 직계존속 또는 그 배우자가 소유하
R3. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=0
    [page 1] [ApplyHome접속] Q. 구글 크롬 브라우저로도 청약할 수 있나요? A. Microsoft Edge, 구글 크롬, 모질라 파이어폭스 브라우저 등에서 청약신청 하실 수 있으며 다른 메뉴도 이용할 수 있습니다.  [page 2] [ApplyHome접속] Q. 보안프로그램을 꼭 설치해야 하나요? A. 청약
R4. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=1
     [page 5] [ApplyHome접속] Q. 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하나요? A. 반드시 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하는 것은 아닙니다. 다른 은행이라도 상관없으나, ①공동인증서(은행보험용 또는 범용) 및 ②금융인증서, ③네이버인증서, ④KB국민인증서, ⑤토스인증서, ⑥신
```
</details>


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _32.6s_

제공된 그래프 컨텍스트만 보면, **부모님 두 분을 모두 부양가족으로 넣을 수 있다고 단정할 근거는 부족합니다.**

그래프에 반복해서 나타나는 패턴은 다음과 같습니다.

- **화면(Screen):** `부양가족`, `민영주택 가점제`, `주택소유여부`, `주택소유 판단기준`
- **작업(Task):** `부양가족 해당 여부와 인원수를 확인한다`
- **설정(Setting):** `부양가족 수`, `직계존속은 일정 요건 충족 시 포함`
- **결과(Outcome):** `가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다`

즉, 이 그래프가 말해주는 핵심은:

1. 민영주택 가점제에서는 **부양가족 수**가 가점 항목입니다.
2. **직계존속(부모님)** 은 **일정 요건을 충족해야** 부양가족에 포함됩니다.
3. 그래서 실제 판단은 **부양가족 해당 여부**와 **주택소유여부 / 주택소유 판단기준**을 함께 확인해야 합니다.

질문하신 사례에서,
- 아버지가 **본인 명의 아파트를 소유**하고 있다는 사실은
- 그래프에 나온 **주택소유 판단기준** 검토 대상이지만,
- 이 컨텍스트만으로는 **그 사실이 어머니까지 포함해 두 분 모두를 부양가족에서 제외하는지**, 혹은 **한 분씩 별도로 판단하는지**가 명시되어 있지 않습니다.

따라서 이 그래프 근거로는 **“부모님 두 분 다 가능하다” 또는 “불가능하다”를 확정할 수 없고**,  
다만 **직계존속 포함 요건과 주택소유 판단기준을 충족하는지 별도로 확인해야 한다**고 답하는 것이 맞습니다.

<details><summary>retrieval log 보기</summary>

```
## Seed Phrase Nodes
1. score=8.00 type=Screen label=부양가족
2. score=7.55 type=Screen label=민영주택 가점제
3. score=7.37 type=Screen label=가점제 재당첨 제한
4. score=6.54 type=Setting label=부양가족 수
5. score=6.36 type=Setting label=가점제 당첨·예비입주자 기준
6. score=5.95 type=Task label=가점제 또는 일반공급에서 무주택기간 산정
7. score=5.92 type=Task label=특별공급 횟수 제한과 가점제 재당첨 제한 해당 여부를 확인
8. score=5.59 type=Setting label=본인 청약통장 가입기간

## Graph Retrieval Summary
expanded_phrase_nodes=8
retrieved_topics=4
analysis_mode=overview
accepted_topics=0
rejected_topics=0
other_topics=4

## Representative Topics
Top graph matches:
  [C1] file=주택청약 FAQ.pdf topic=6 outcome=가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다 score=22.09
       작업/질문: 민영주택 가점제에서는 무주택기간과 부양가족은 어떻게 계산하나요?
       결론: 가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다.
  [C2] file=주택청약 FAQ.pdf topic=8 outcome=당첨자 관리 등록 및 재당첨 제한 적용 여부가 결정된다 score=19.65
       작업/질문: 당첨되면 당첨자로 관리되거나 재당첨 제한을 받나요?
       결론: 당첨자 관리 등록 및 재당첨 제한 적용 여부가 결정된다.
  [C3] file=주택청약 FAQ.pdf topic=5 outcome=무주택기간이 산정되어 가점 또는 자격 판단에 반영된다 score=5.95
       작업/질문: 주택을 처분했거나 분양권을 취득한 시점에 따라 무주택기간은 어떻게 계산하나요?
       결론: 무주택기간이 산정되어 가점 또는 자격 판단에 반영된다.
  [C4] file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=10 outcome=민영주택 청약가점을 확인할 수 있다 score=5.59
       작업/질문: 민영주택 가점제는 어떤 항목으로 계산되나요?
       결론: 가점 항목을 바탕으로 민영주택 청약가점을 확인할 수 있다.
```
</details>


---

### ✅ 파이프라인 4 · Search Agent (multi-step)  · _67.9s_

## User Query
본인이 세대주이고 만 65세인 아버지와 62세인 어머니를 4년째 같은 주민등록표에 모시고 있습니다. 그런데 아버지가 본인 명의의 아파트를 한 채 소유하고 계십니다. 가점제 청약 시 부모님 두 분 다 제 부양가족으로 점수를 받을 수 있나요?

## Step 1: search_topics
reason: 부모를 부양가족으로 인정하는 기본 요건과, 직계존속의 연령·동일 주민등록표 요건, 그리고 부모 중 한 명이 주택을 소유한 경우 각각의 인정 여부를 판단하는 기준을 확인하기 위해
query: 가점제 부양가족 직계존속 부모 동일 주민등록표 3년 이상 60세 이상 주택 소유 배우자 부모 각각 부양가족 인정 여부
S1-1. topic score=36.646 file=청약홈 FAQ - 03.청약가점제 topic=2
   질문: 청약가점제에서 부양가족수는 누구를 어떻게 인정하나요?
   결과: 인정되는 부양가족 수에 따라 가점이 산정된다.
S1-2. topic score=24.929 file=청약홈 FAQ - 03.청약가점제 topic=3
   질문: 청약가점제에서 무주택기간과 무주택자 여부는 어떻게 판단하나요?
   결과: 무주택자 해당 여부와 무주택기간이 산정되어 가점제 점수에 반영된다.
S1-3. topic score=21.165 file=주택청약 FAQ.pdf topic=6
   질문: 민영주택 가점제에서는 무주택기간과 부양가족은 어떻게 계산하나요?
   결과: 가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다.
S1-4. topic score=11.339 file=주택청약 FAQ.pdf topic=3
   질문: 무주택세대구성원은 어떻게 판단하나요?
   결과: 세대원 전체 기준으로 무주택세대구성원 해당 여부가 결정된다.

## Step 2: search_raw
reason: 부모 중 한 명이 주택을 소유한 경우 가점제 부양가족 인정에서 누가 제외되는지, 그리고 다른 부모는 별도로 인정되는지 원문 규정을 확인하기 위해
query: 부양가족 직계존속 주택소유 제외 아버지 본인 명의 아파트 소유 어머니는 별도 인정 동일 주민등록표 3년 이상 가점제
S2-1. raw score=38.099 file=청약홈 FAQ - 03.청약가점제 chunk=1
   주민등록표등본에 등재된 직계존속) 단, 직계존속 및 그 배우자 중 한 명이라도 주택을 소유(분양권 등 포함)하고 있는 경우 부양가족으로 보지 않으나 공급규칙 제53조에 해당하는 주택을 가지고 있는 경우 아래와 같이 판단 *주택공급에 관한 규칙 제53조(주택소유여부 판정기준) 적용 사례 - 직계존속 또는 그 배우자가 소유하는 주택이 소형,저가주택 등(제9호) 및 제53조 각 호(제6호 제외)에
S2-2. raw score=32.333 file=청약홈 FAQ - 03.청약가점제 chunk=2
   도 부양가족 아버지 또는 어머니 -청약신청자(분리세대인 배우자)가 세대주이고, 부모님이 3년 이상 동일한 주민등록표등본에 등재된 경우 부양가족 배우자의 아버지 또는 어머니 -배우자(분리세대인 신청자)가 세대주이고, 배우자의 부모님이 3년 이상 동일한 주민등록표등본에 등재된 경우 부양가족 형 또는 동생 부양가족 아님 처제 아들 또는 딸 -미혼이고 만30세 미만인 경우 부양가족 -미혼이고 만30
S2-3. raw score=30.515 file=청약 자격 체크리스트 (applyhome_checklist) chunk=4
   제자매 및 동거인은 세대원에서 제외.  [homeless_period_calculation] - basis: 청약신청자와 그 배우자를 기준으로 계산 (외국인은 계산 불가하여 0점 처리) - scenarios:   * past_ownership: 주택 소유 이력 없음, marriage_before_30: 만 30세 이전 미혼, calculation_period: 만 30세가 된 날 ~ 입주자 
S2-4. raw score=23.786 file=주택청약 FAQ.pdf chunk=161
   경우에는 계속하여 같은 세대별 주민등록표상에 등재된 것으로 볼 수 없으므로 노부모  부양 특별공급 신청 자격이 제한되며, 이에 대한 확인을 위해 증빙서류 제출 시 출입국 사실증명원을 제출하여 해외 체류여부를 증명하여야 합니다. 256 노부모 특별공급 청약 시 청약신청자의 어머니는 청약자와 동일한 주민등록에  등재되어 있으나 아버지는 주민등록이 분리되어 있는 경우, 아버지도 무주택이어야  하는

## Step 3: finish
reason: 직계존속과 그 배우자 중 한 명이라도 주택을 소유하면 부양가족으로 보지 않는다는 원문 근거를 이미 확인했으므로, 추가 검색 없이 최종 답변 단계로 넘어가도 충분합니다.

## Stop Reason
planner selected finish at step 3

## Final Answer
결론부터 말씀드리면, **아버지가 일반적인 주택 1채를 소유하고 계신 경우에는 부모님 두 분을 모두 가점제 부양가족으로 인정받기는 어렵습니다**. 직계존속은 신청자와 3년 이상 같은 주민등록표등본에 등재되어 있어야 하지만, **직계존속과 그 배우자 중 한 명이라도 주택을 소유하고 있으면 부양가족으로 보지 않는다**고 되어 있습니다.[S1-1][S2-1]

특히 **만 60세 이상 직계존속이 주택을 소유한 경우**에도 원칙적으로는 **부양가족 수에 포함하지 않습니다**. 다만 그 주택이 **주택공급에 관한 규칙 제53조의 나머지 예외에 해당하는 경우에는 인정될 수 있다**고 되어 있습니다.[S2-1]

따라서 지금 사례에서는  
- **아버지**: 65세이고 주택 소유가 있으므로, 원칙적으로 부양가족 인정이 안 됩니다.[S2-1]  
- **어머니**: 부모님이 같은 주민등록표에 4년째 함께 계시더라도, **직계존속과 그 배우자 중 한 명이라도 주택을 소유하면 부양가족으로 보지 않는다**는 기준 때문에 함께 인정되기 어렵습니다.[S2-1]

즉, **아버지의 아파트가 예외적으로 제53조에 해당하지 않는 한, 부모님 두 분 모두 점수에 반영되기는 어렵습니다**.[S2-1]

주의할 점은, 질문의 핵심은 **같은 주민등록표 등재 기간만으로 끝나지 않고, 직계존속 및 그 배우자의 주택소유 여부까지 함께 본다**는 점입니다.[S1-1][S2-1]


---


### 17-3. 파이프라인 3 지향 질문 (Graph RAG가 필요한 유형)

In [34]:
results3 = compare(Q3, show_logs=True)

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Topic-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...
▶ 파이프라인 4 · Search Agent (multi-step) 실행 중...


### 질문

> 현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이 원천적으로 차단되는 지역이나 주택 유형은 무엇이며, 반대로 다주택자도 1순위 당첨을 노려볼 수 있는 규제 패턴(지역 및 추첨제 비율)은 어떻게 연결되어 있나요?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _32.8s_

다주택자(세대 기준 2주택 이상)가 **일반공급 1순위에서 원천적으로 막히는 대표 구간**은 다음과 같습니다.  
- **투기과열지구·청약과열지역의 민영주택**: 2주택 이상을 소유한 세대에 속한 자는 일반공급 1순위가 제한되어 2순위로 청약해야 합니다.[1][3][5]  
- **수도권에 지정된 공공주택지구(개발제한구역 해제 면적 50% 이상)의 민영주택**: 이 구간도 2주택 이상 보유 세대는 1순위 제한 대상입니다.[1][5]  
- **85㎡ 초과 공공건설임대주택**: 2주택 이상 보유 세대는 1순위가 제한됩니다.[1][5]  

반면, **국민주택**은 질문의 “2주택 이상 보유” 자체보다 **무주택세대구성원 여부**와 세대주 요건이 핵심이라서, 규정상 2주택자 차단 구조와는 다르게 설명됩니다.[1]

다주택자가 “1순위 당첨”을 노려볼 수 있는 규제 패턴은, **규제가 약한 지역일수록 추첨제 비중이 커지는 구조**와 연결됩니다.  
- 민영주택 일반공급의 가점제 비율은 **투기과열지구 85㎡ 이하 100%, 85㎡ 초과 50%**, **청약과열지역 85㎡ 이하 75%, 85㎡ 초과 30%**, **그 외 지역 40% 이하(지자체장 결정)**로 제시되어 있습니다.[4]  
- 즉, **투기과열지구·청약과열지역처럼 규제가 강한 곳은 가점제 비중이 높고 2주택 세대는 1순위 자체가 막히는 경우가 많아** 다주택자에게 불리합니다.[1][4][5]  
- 반대로 **그 외 지역의 민영주택은 2주택 이상 보유 세대에 대한 1순위 제한 문구가 제시되지 않아**, 상대적으로 1순위 청약이 가능한 범위가 넓습니다. 다만 이 경우에도 **추첨제 우선공급은 무주택세대구성원에게 먼저 75%를 배정**하고, 남은 물량도 **무주택세대구성원과 1주택 세대**를 대상으로 하므로, **2주택자는 추첨제 우선순위에서도 제외**됩니다.[3][4]

정리하면, **“2주택 이상 세대의 1순위 차단”은 규제지역·특정 공공지구·85㎡ 초과 공공건설임대에서 강하게 적용**되고, **규제가 약한 지역의 민영주택일수록 가점제 비중이 낮아 추첨 비중이 커지지만, 추첨제도 다주택자에게는 직접적인 우호 규칙이 아닙니다**.[1][3][4][5]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=23.006 file=청약홈 FAQ - 04.청약 제한 chunk=8
    계약체결일 * 단, 최초로 예비입주자를 입주자로 선정하는 경우로서 동·호수 배정의 추첨에 참가하여 동·호수를 배정받은 경우 : 동·호수배정일 (계약여부와 관계 없이 당첨자 관리) 4. 조합주택(직장, 지역)의 조합원 : 사업계획승인일 5. 조합주택(재건축, 재개발)의 조합원 : 관리처분계획인가일. 단, 2006년 8월 18일 이전 사업승인을 받은 재건축조합원의 경우 ‘사업계획승인일’ 기준으
2. score=22.522 file=주택청약 FAQ.pdf chunk=138
   택세대구성원 요건을 충족하지  않은 것인가요?    「주택공급에 관한 규칙」 부칙 <국토교통부령 제565호, 2018.12.11.>에 따라 혼인신고일  이후에 주택을 보유하였더라도 아래 요건을 모두 충족하는 경우에는 신혼부부 주택 특별 공급의 2순위로 자격을 부여합니다.  ① 2018.12.11. 이전에 기존 소유한 주택을 처분할 것 ② 기존 소유 주택의 처분일부터 계속해서 무주택세대구성원을
3. score=19.471 file=청약홈 FAQ - 04.청약 제한 chunk=6
   역) * 국토교통부공고 제2022-1408호 참조 · 2023.01.05. 시행(해제) : - 서울특별시 도봉·강북·노원·성북·은평·종로·중랑·동대문·서대문·중·마포·성동·광진·강서·양천·구로·영등포·금천·동작·관악·강동구 - 경기도 과천시, 성남시 분당·수정구, 하남시, 광명시 * 국토교통부공고 제2023-2호 참조 · 2025.10.16. 시행(지정) : - 서울특별시 도봉·강북·노원·성
4. score=19.222 file=주택청약 FAQ.pdf chunk=119
   3년 이상 무주택세대구성원으로 납입횟수가 많은 자  (동일횟수 시 추첨) 40㎡초과 • 3년 이상 무주택세대구성원으로  저축총액이 많은 자  (동일총액 시 추첨) 민영주택 가점제 지역별 지정 • 가점제 점수가 높은 순 (동일점수 시 추첨)   * 가점제 적용비율 구 분 85㎡이하 85㎡초과 수도권 공공택지 100% 50% 이하에서 지자체장 결정 투기과열지구 100% 50% 청약과열지역 75%
5. score=18.910 file=청약홈 FAQ - 02.청약통장 chunk=1
   입] Q. 과거 아파트 당첨사실이 있는데 청약통장에 신규 가입하면 1순위가 될 수 있나요? A. 네. 1순위 청약신청 자격이 되면 가능합니다. * 청약통장을 사용하여 분양주택 또는 분양전환공공임대주택의 입주자로 선정된 경우, 동일한 통장으로 다른 주택에 청약할 수 없습니다. 과거 당첨된 청약통장을 해지한 후 신규 가입하셔야 합니다. 다만, 아래에 해당하는 경우에는 일반공급 1순위 청약이 제한
6. score=18.832 file=주택청약 FAQ.pdf chunk=25
    구분되나요? ·······104 Q198. 특별공급을 부부가 중복해서 각각 청약 신청이 가능한가요? ··································105 Q199. 자녀수 산정은 어떻게 하나요?(초혼인 경우) ··························································105 Q200. 재혼한 경우에 자녀수는 어떻게 산정하나요? ··
7. score=18.397 file=주택공급에 관한 규칙.pdf chunk=132
   유한 세대에 속한 사 람이 아님(1주택을 소유한 세대에 속한 사람은 추첨제 적용) [  ]입주자모집공고일 현재 재당첨 제한 대상자의 세대에 속한 사 람이 아님 2) 1)에 해당하지 않는 경우 로 공공주택지구 내 민영주 택 또는 공공주택지구 외  85제곱미터 초과 공공건설 임대주택 [  ]입주자모집공고일 현재 2주택 이상을 소유한 세대에 속하지  않음(1주택을 소유한 세대에 속한 사람은 추첨제
8. score=17.824 file=주택청약 FAQ.pdf chunk=141
   우자와 같은 세대별 주민등록표에  등재되어 있는 사람        * 라. 주택공급신청자의 직계비속(직계비속의 배우자를 포함, 이하 같음)으로서 주택 공급신청자 또는 주택공급신청자의 배우자와 세대별 주민등록표에 함께 등재 되어 있는 사람        * 마. 주택공급신청자의 배우자의 직계비속으로서 주택공급신청자와 세대별 주민 등록표에 함께 등재되어 있는 사람  [page 139] Ⅲ. 특별공
9. score=16.697 file=주택청약 FAQ.pdf chunk=205
   확인은 어떻게 진행되는지?    사업주체는 입주대상자로 선정된 자에 대한 주택소유여부 등의 전산검색을 국토부장관 에게 의뢰하여야 하며, 주택청약업무수행기관(한국부동산원)을 통해 전달받은 주택소유  검색결과, 세대원 및 청약 제한사항 관련 정보와 당첨자 및 예비입주자로부터 제출받은  관련 서류를 통해 자격을 확인하여야 합니다. 전산검색 및 제출서류 확인 결과 공급자격 또는 선정순위를 달리한 부
10. score=15.570 file=주택청약 FAQ.pdf chunk=15
    지역주택조합 조합원도 당첨자로 관리되는지? ·························································61 Q124. 관리처분계획인가일 이후 정비사업 입주권, 사업계획승인일 이후 지역주택조합 입주권을  양도받은 자도 당첨자로 관리되는지? ······························································
```
</details>


---

### ✅ 파이프라인 2 · Topic-based (structural)  · _39.2s_

다주택자라고 해서 **모든 1순위 청약이 전면 차단되는 것은 아니고**, 특정 **지역·주택유형**에서만 1순위 제한이 걸립니다. 구체적으로는 **투기과열지구, 청약과열지역, 수도권에 지정된 공공주택지구(개발제한구역 해제 면적 50% 이상), 85㎡ 초과 공공건설임대주택**에 청약하는 경우, **2주택 이상을 소유한 세대에 속한 자는 일반공급 1순위가 제한**됩니다. [R6]

반대로 말하면, 위에 열거된 범위가 아닌 주택·지역에서는 다주택자라는 이유만으로 1순위가 자동 배제된다고 보기는 어렵고, **청약통장·가입기간·납입횟수·예치금액 등 해당 주택의 순위 요건**을 충족하면 1순위 청약 가능성을 따질 수 있습니다. [R3][R6]

다주택자가 **1순위 당첨을 노려볼 수 있는 핵심 패턴**은 **민영주택 일반공급의 가점제+추첨제 구조**입니다. 민영주택 일반공급은 전부 가점제가 아니라 **일정 비율은 가점제, 나머지는 추첨제**로 선정하므로, 가점 경쟁이 불리해도 **추첨 물량을 노리는 전략**이 가능합니다. [R1]

지역·면적별로 보면, **85㎡ 초과 민영주택**은  
- **수도권·공공주택지구·투기과열지구: 가점제 100%**  
- **청약과열지구: 가점제 80%**  
- **그 외 지역: 가점제 50%**  
로 안내되어 있어, 이 구간에서는 특히 규제지역일수록 추첨 몫이 거의 없거나 적습니다. [R1]

반면 **85㎡ 이하**는 추첨 비중이 남습니다. 안내표에 따르면 **60㎡ 이하**는 수도권·공공주택지구·투기과열지구에서 **가점제 40%**, **60㎡ 초과~80㎡ 이하**는 같은 지역에서 **가점제 70%**이며, **청약과열지구와 그 외 지역은 가점제 비율이 더 낮거나(40% 이하) 시장·군수·구청장이 정하는 범위**로 운영됩니다. 즉, 다주택자는 **규제 강도가 낮은 지역이나 추첨 비율이 남는 면적대**에서 1순위 당첨을 노리는 구조입니다. [R1]

정리하면, **다주택자 1순위 원천 제한이 강한 패턴**은 “**투기과열지구·청약과열지역·특정 수도권 공공주택지구·85㎡ 초과 공공건설임대주택**”이고, **당첨 기회가 남는 패턴**은 “**그 밖의 지역의 민영주택 일반공급, 특히 추첨제가 남는 면적대**”입니다. [R6][R1]

다만, **실제 신청 가능 여부는 해당 단지의 입주자모집공고에서 지역, 주택유형, 면적, 공급방식, 세대주 요건, 청약통장 순위 요건을 함께 확인해야** 합니다. [R3][R6]

<details><summary>retrieval log 보기</summary>

```
## Retrieved Topics
1. score=14.041 file=청약홈 FAQ - 03.청약가점제 topic=1
   질문: 청약가점제 적용비율은 면적과 지역에 따라 어떻게 달라지나요?
   결과: 해당 주택 유형에서 가점제와 추첨제의 적용 비율을 확인할 수 있다.
2. score=11.016 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=7
   질문: 수도권 외 거주자인데 1순위가 아닌 2순위로 조회되는 이유는 무엇인가요?
   결과: 실제 청약 주택 기준으로 1순위 여부를 확인할 수 있다.
3. score=10.968 file=청약홈 FAQ - 02.청약통장 topic=5
   질문: 청약통장으로 1순위가 되려면 어떤 기준을 충족해야 하나요?
   결과: 지역·주택유형·납입요건을 충족하면 1순위 또는 2순위로 청약할 수 있다.
4. score=10.264 file=청약홈 FAQ - 02.청약통장 topic=6
   질문: 과거에 당첨된 적이 있거나, 여러 지역에서 청약하려는 경우 어떻게 되나요?
   결과: 당첨 이력, 지역 요건, 중복 청약 규정을 검토해 청약 가능 여부와 유효 당첨이 결정된다.
5. score=7.576 file=주택청약 FAQ.pdf topic=7
   질문: 특별공급 중 신혼부부나 생애최초 특별공급 자격은 어떻게 판단하나요?
   결과: 요건 충족 시 신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부가 결정된다.

## Added Raw Chunks
R1. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=0
    [page 1] [청약가점제 내용] Q. 청약가점제란? A. 가점항목에 대하여 산정한 점수(가점제 점수)가 높은 순으로 입주자를 선정하는 방식을 말합니다. 민영주택 일반공급 입주자 선정 시 일정비율(나머지 비율은 추첨으로 선정)을 가점제로 선정합니다. □ 가점항목 항 목 점 수 범 위 무주택기간 32점 1년미만~15년이상
R2. score=0.000 file=청약홈 FAQ - 03.청약가점제 chunk=1
    주민등록표등본에 등재된 직계존속) 단, 직계존속 및 그 배우자 중 한 명이라도 주택을 소유(분양권 등 포함)하고 있는 경우 부양가족으로 보지 않으나 공급규칙 제53조에 해당하는 주택을 가지고 있는 경우 아래와 같이 판단 *주택공급에 관한 규칙 제53조(주택소유여부 판정기준) 적용 사례 - 직계존속 또는 그 배우자가 소유하
R3. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=0
    [page 1] [ApplyHome접속] Q. 구글 크롬 브라우저로도 청약할 수 있나요? A. Microsoft Edge, 구글 크롬, 모질라 파이어폭스 브라우저 등에서 청약신청 하실 수 있으며 다른 메뉴도 이용할 수 있습니다.  [page 2] [ApplyHome접속] Q. 보안프로그램을 꼭 설치해야 하나요? A. 청약
R4. score=0.000 file=청약홈 FAQ - 01.청약이 잘 안되세요? chunk=1
     [page 5] [ApplyHome접속] Q. 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하나요? A. 반드시 청약통장 가입은행의 인터넷뱅킹에 가입되어 있어야 하는 것은 아닙니다. 다른 은행이라도 상관없으나, ①공동인증서(은행보험용 또는 범용) 및 ②금융인증서, ③네이버인증서, ④KB국민인증서, ⑤토스인증서, ⑥신
R5. score=0.000 file=청약홈 FAQ - 02.청약통장 chunk=0
    [page 1] [청약통장의 가입] Q. 청약통장이란? 청약통장의 종류는? A. 청약통장이란? - 청약통장(입주자저축)은 주택법에 따라 주택을 공급받으려는 자에게 입주금의 일부나 전부를 저축하도록 하는 제도입니다. - 특별공급(기관추천 특별공급 중 국가유공자, 장애인, 철거민 등은 청약통장 가입 불필요)·일반공급1·2순위
R6. score=0.000 file=청약홈 FAQ - 02.청약통장 chunk=1
    입] Q. 과거 아파트 당첨사실이 있는데 청약통장에 신규 가입하면 1순위가 될 수 있나요? A. 네. 1순위 청약신청 자격이 되면 가능합니다. * 청약통장을 사용하여 분양주택 또는 분양전환공공임대주택의 입주자로 선정된 경우, 동일한 통장으로 다른 주택에 청약할 수 없습니다. 과거 당첨된 청약통장을 해지한 후 신규 가입하셔
R7. score=0.000 file=주택청약 FAQ.pdf chunk=0
    [page 1] [추가 정정]  [page 2] 본 FAQ는 2022년 7월 기준으로 작성되었으며, 관련 법령의 개정, 법령해석  변경 등에 따라 변경될 수 있음을 알려드리며, 참고용으로만 활용하여 주시기  바랍니다.   또한 개별 사실관계의 변동 등으로 인한 유사사례인 경우에 다른 해석이 있을  수 있으며, 본 FAQ를
R8. score=0.000 file=주택청약 FAQ.pdf chunk=1
    ························8 Q6. 「주택공급에 관한 규칙」 제34조에 따른 대규모택지개발지구에서 주택이 공급되는 경우  일반공급 뿐만 아니라 특별공급 물량도 공급비율에 따라 배정되는지? ···························8 Q7. 행정중심복합도시예정지역에서 공급하는 주택의 경우 공급비율
```
</details>


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _35.7s_

제공된 graph context만 기준으로 보면, **“2주택 보유자라서 어느 지역·어느 주택유형에서 1순위가 원천 차단된다”**를 단정할 만한 근거는 **부족합니다**. 다만 반복적으로 연결되는 패턴은 분명합니다.

## 1) 1순위 판단은 “다주택 여부” 하나로 끝나지 않음
graph에서는 1순위 관련 판단이 다음 축으로 나뉘어 있습니다.

- **실제 청약하려는 주택 기준의 순위 조건**
- **해당 주택건설지역 거주 여부 / 거주기간**
- **입주자저축 가입기간·납입횟수**
- **민영주택의 가점제**
- **무주택기간, 부양가족 수**

즉, 1순위는 “내가 집을 몇 채 갖고 있느냐”보다 **신청 주택 기준의 순위요건 + 지역요건 + 가점요건**의 결합으로 판단됩니다.

## 2) 다주택자에게 직접적으로 연결된 “제한” 패턴
context에서 직접 확인되는 제한 연결은 주로 **“일반공급 1순위 및 투기과열지구 제한 확인”**입니다.  
여기서 드러나는 패턴은:

- **일반공급 1순위**
- **투기과열지구 제한**
- 그리고 별도로 **특별공급(신혼부부, 생애최초)** 자격 판단

즉, 다주택자라는 이유로 전 지역·전 주택유형에서 일괄 차단된다고 나오지는 않고, **특히 투기과열지구에서 일반공급 1순위 제한을 추가로 확인하는 구조**로 보입니다.

## 3) “원천 차단”보다 “경쟁력 약화”에 가까운 패턴
민영주택 가점제 관련 graph 패턴은 다음과 같습니다.

- **무주택기간**
- **부양가족 수**
- **가점 산정기준**
- **주택소유여부 / 주택소유 판단기준**
- **가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영**

여기서 중요한 점은, context상 다주택자는 보통 **무주택기간 산정에서 불리**하고, **가점제 점수에서도 불리**할 가능성이 높다는 점입니다.  
하지만 이것은 **“1순위 자격 자체의 전면 차단”**이라기보다, **1순위 당첨 가능성을 낮추는 구조**에 가깝습니다.

## 4) 다주택자도 1순위 당첨을 노려볼 수 있는 연결 구조
graph가 보여주는 연결은 다음과 같습니다.

### A. 지역 조건
- **해당지역 거주여부**
- **해당지역 거주기간**
- **해당지역 우선공급 또는 일반 신청 가능 여부**

즉, 다주택자라도 **해당지역 거주요건**을 맞추면 신청 구조 자체는 열릴 수 있습니다.

### B. 주택 유형 조건
- **민영주택 가점제**
- **무주택기간**
- **부양가족 수**

다주택자는 무주택기간 측면에서 불리할 수 있지만, context상 **민영주택에서는 가점제 점수 구조 안에서 경쟁**하게 됩니다.  
즉, **완전 배제보다 점수 경쟁**의 구조입니다.

### C. 규제 패턴의 핵심
정리하면, graph에서 보이는 패턴은:

- **투기과열지구 + 일반공급 1순위 제한 확인**
- **해당지역 거주요건 충족 여부 확인**
- **민영주택 가점제에서 무주택기간·부양가족 수 반영**

이 연결 때문에, 다주택자는  
**“투기과열지구에서는 제한이 더 강하게 작동할 수 있지만, 모든 지역에서 1순위가 원천 차단된다고는 볼 수 없고”**,  
반대로 **지역요건과 공급유형, 가점제 구조에 따라 1순위 당첨을 시도하는 패턴이 남아 있다**고 해석할 수 있습니다.

## 5) 근거가 부족한 부분
제공된 context에는 다음이 **명시되어 있지 않습니다**.

- “2주택 이상이면 1순위가 불가능한 지역 목록”
- “민영/국민주택별로 다주택자 차단 여부”
- “투기과열지구 내에서도 어떤 주택형의 추첨제 비율이 몇 %인지”
- “추첨제에서 다주택자가 얼마만큼 기회를 갖는지”

따라서 **지역명과 추첨제 비율까지 포함한 정확한 매핑은 이 graph만으로는 확정할 수 없습니다**.

요약하면, 이 graph의 패턴은 **다주택자 전면 차단**이 아니라,  
**투기과열지구·해당지역 거주요건·민영주택 가점제(무주택기간/부양가족)**의 조합으로 1순위 가능성이 갈리는 구조입니다.

<details><summary>retrieval log 보기</summary>

```
## Seed Phrase Nodes
1. score=13.86 type=Task label=일반공급 1순위 및 투기과열지구 제한 확인
2. score=13.22 type=Task label=1순위 산정 기준 확인
3. score=6.54 type=Setting label=부양가족 수
4. score=6.29 type=Setting label=해당 지역 거주여부
5. score=5.12 type=Task label=신청자의 현재 거주지가 해당지역인지 확인
6. score=4.74 type=Setting label=주택 취득일
7. score=4.62 type=Task label=입주자모집공고일 현재 가입 여부 확인
8. score=4.62 type=Setting label=연락주소는 등본과 다를 수 있음

## Graph Retrieval Summary
expanded_phrase_nodes=8
retrieved_topics=7
analysis_mode=overview
accepted_topics=0
rejected_topics=0
other_topics=7

## Representative Topics
Top graph matches:
  [C1] file=주택청약 FAQ.pdf topic=7 outcome=신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부 결정 score=13.86
       작업/질문: 특별공급 중 신혼부부나 생애최초 특별공급 자격은 어떻게 판단하나요?
       결론: 요건 충족 시 신혼부부 또는 생애최초 특별공급 신청 및 당첨 가능 여부가 결정된다.
  [C2] file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=7 outcome=실제 청약 주택 기준으로 1순위 여부를 확인할 수 있다 score=13.22
       작업/질문: 수도권 외 거주자인데 1순위가 아닌 2순위로 조회되는 이유는 무엇인가요?
       결론: 실제 청약 주택 기준으로 1순위 여부를 확인할 수 있다.
  [C3] file=주택청약 FAQ.pdf topic=0 outcome=거주지역과 거주기간 요건에 따라 해당지역 우선공급 또는 일반 신청 가능 여부가 결정된다 score=11.41
       작업/질문: 해당 주택건설지역에 거주하지 않아도 청약신청이 가능한가요?
       결론: 거주지역과 거주기간 요건에 따라 해당지역 우선공급 또는 일반 신청 가능 여부가 결정된다.
  [C4] file=주택청약 FAQ.pdf topic=6 outcome=가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다 score=6.54
       작업/질문: 민영주택 가점제에서는 무주택기간과 부양가족은 어떻게 계산하나요?
       결론: 가점제 점수가 산정되어 민영주택 1순위 당첨 가능성에 반영된다.
  [C5] file=주택청약 FAQ.pdf topic=5 outcome=무주택기간이 산정되어 가점 또는 자격 판단에 반영된다 score=4.74
       작업/질문: 주택을 처분했거나 분양권을 취득한 시점에 따라 무주택기간은 어떻게 계산하나요?
       결론: 무주택기간이 산정되어 가점 또는 자격 판단에 반영된다.
```
</details>


---

### ⚠️ 파이프라인 4 · Search Agent (multi-step)  · _33.3s_

## User Query
현재 2주택을 소유한 다주택자입니다. 다주택자라는 이유로 1순위 청약이 원천적으로 차단되는 지역이나 주택 유형은 무엇이며, 반대로 다주택자도 1순위 당첨을 노려볼 수 있는 규제 패턴(지역 및 추첨제 비율)은 어떻게 연결되어 있나요?

## Step 1: search_topics
reason: 다주택자의 1순위 청약이 원천 차단되는 지역·주택유형과, 반대로 1순위 당첨 가능성이 남는 추첨제 비율/규제 조합의 구조를 먼저 정리해야 합니다.
query: 다주택자 1순위 청약 제한 지역 주택유형 원천 차단 추첨제 비율 규제 패턴 조정대상지역 투기과열지구 민영주택 국민주택
S1-1. topic score=14.567 file=청약홈 FAQ - 02.청약통장 topic=5
   질문: 청약통장으로 1순위가 되려면 어떤 기준을 충족해야 하나요?
   결과: 지역·주택유형·납입요건을 충족하면 1순위 또는 2순위로 청약할 수 있다.
S1-2. topic score=13.826 file=청약홈 FAQ - 03.청약가점제 topic=1
   질문: 청약가점제 적용비율은 면적과 지역에 따라 어떻게 달라지나요?
   결과: 해당 주택 유형에서 가점제와 추첨제의 적용 비율을 확인할 수 있다.
S1-3. topic score=11.152 file=청약홈 FAQ - 02.청약통장 topic=6
   질문: 과거에 당첨된 적이 있거나, 여러 지역에서 청약하려는 경우 어떻게 되나요?
   결과: 당첨 이력, 지역 요건, 중복 청약 규정을 검토해 청약 가능 여부와 유효 당첨이 결정된다.
S1-4. topic score=8.053 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=7
   질문: 수도권 외 거주자인데 1순위가 아닌 2순위로 조회되는 이유는 무엇인가요?
   결과: 실제 청약 주택 기준으로 1순위 여부를 확인할 수 있다.

## Step 2: search_raw
reason: 이전 topic 검색으로 1순위 요건과 가점·추첨제 구조는 잡혔으므로, 이제 원문 규정에서 다주택자(유주택자) 1순위 원천 제한 조항과 예외적으로 추첨제 기회가 남는 지역·주택유형의 비율 문구를 직접 확인해야 합니다.
query: 다주택자 1순위 청약 제한 투기과열지구 청약과열지역 국민주택 민영주택 무주택세대구성원 추첨제 비율 조정대상지역
S2-1. raw score=23.769 file=청약홈 FAQ - 04.청약 제한 chunk=6
   역) * 국토교통부공고 제2022-1408호 참조 · 2023.01.05. 시행(해제) : - 서울특별시 도봉·강북·노원·성북·은평·종로·중랑·동대문·서대문·중·마포·성동·광진·강서·양천·구로·영등포·금천·동작·관악·강동구 - 경기도 과천시, 성남시 분당·수정구, 하남시, 광명시 * 국토교통부공고 제2023-2호 참조 · 2025.10.16. 시행(지정) : - 서울특별시 도봉·강북·노원·성
S2-2. raw score=23.506 file=주택청약 FAQ.pdf chunk=108
   획인가의  취소 등으로 사실상 주택을 공급받을 수 없게 된 자      ⑥ 제58조에 따라 당첨이 취소된 부적격 당첨자 위 요건에 해당하지 않는 경우에는 당첨자 명단 삭제가 불가하며, 당첨된 이후 ①~③  요건에 해당하게 된 경우 사업주체에게 관련서류를 제출하여 당첨자 명단 삭제를 요청 하여야 하며, 사업주체는 삭제 요건에 해당하는 경우 주택청약업무수행기관(한국부동산 원)에 통보하여 당첨자 
S2-3. raw score=23.219 file=주택청약 FAQ.pdf chunk=119
   3년 이상 무주택세대구성원으로 납입횟수가 많은 자  (동일횟수 시 추첨) 40㎡초과 • 3년 이상 무주택세대구성원으로  저축총액이 많은 자  (동일총액 시 추첨) 민영주택 가점제 지역별 지정 • 가점제 점수가 높은 순 (동일점수 시 추첨)   * 가점제 적용비율 구 분 85㎡이하 85㎡초과 수도권 공공택지 100% 50% 이하에서 지자체장 결정 투기과열지구 100% 50% 청약과열지역 75%
S2-4. raw score=21.247 file=청약홈 FAQ - 04.청약 제한 chunk=8
   계약체결일 * 단, 최초로 예비입주자를 입주자로 선정하는 경우로서 동·호수 배정의 추첨에 참가하여 동·호수를 배정받은 경우 : 동·호수배정일 (계약여부와 관계 없이 당첨자 관리) 4. 조합주택(직장, 지역)의 조합원 : 사업계획승인일 5. 조합주택(재건축, 재개발)의 조합원 : 관리처분계획인가일. 단, 2006년 8월 18일 이전 사업승인을 받은 재건축조합원의 경우 ‘사업계획승인일’ 기준으로

## Step 3: finish
reason: 이미 관련 근거가 충분합니다. 청약과열지역/투기과열지구에서 민영주택·국민주택의 1순위 제한과, 민영주택의 가점제·추첨제 비율 및 추첨 배정 방식까지 확인되어 최종 정리 단계로 넘어갈 수 있습니다.

## Stop Reason
planner selected finish at step 3


[stderr]
Traceback (most recent call last):
  File "D:\snu_python\graphrag\scripts_home\15_search_agent_demo.py", line 295, in <module>
    main()
  File "D:\snu_python\graphrag\scripts_home\15_search_agent_demo.py", line 283, in main
    answer = call_llm(
             ^^^^^^^^^
  File "D:\snu_python\graphrag\scripts_home\15_search_agent_demo.py", line 127, in call_llm
    response = client.chat.completions.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_utils\_utils.py", line 287, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\resources\chat\completions\completions.py", line 1147, in create
    return self._post(
           ^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_base_client.py", line 1259, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dell\miniforge3\envs\openai\Lib\site-packages\openai\_base_client.py", line 1047, in request
    raise self._make_status_error_from_response(err.response) from None
openai.BadRequestError: Error code: 400 - {'error': {'message': 'Selected model is at capacity. Please try a different model.'}}


---


### 17-d. 성능지표 계산

`results1/2/3`에 대해 `compute_metrics(...)`로 score 분포·확신도와 문서 겹침도(Jaccard)를 계산합니다.

In [35]:
metrics1 = compute_metrics(results1, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=21.083 file=청약홈 FAQ - 02.청약통장 chunk=4
   변경] Q. 경기도 거주자로서 수도권 거주 자격으로 서울에 공급되는 주택에 청약하고 싶습니다. 청약예금을 서울 지역 예치금으로 증액해야 하나요? A. 아닙니다. 수도권 거주자 자격으로 청약신청하는 경우 입주자모집공고일 기준 본인이 거주하는 지역(그 밖의 광역시)의 예치금액을 충족하고 신청가능 순위에 해당하면 가능합니다. * 청약예금 및 부금 가입자의 경우 금액 및 면적 변경은 입주자모집공고일
2. score=18.834 file=주택청약 FAQ.pdf chunk=74
    주택 구입 시 : ’22년 말까지 우대이율 적용 31 제2순위로 당첨된 경우 해당 통장은 재사용이 가능한지?  

-- P2 retrieval log 일부 (파싱된 15건) --
## Retrieved Topics
1. score=10.228 file=청약홈 FAQ - 02.청약통장 topic=0
   질문: 청약통장이란 무엇이고, 어떤 종류가 있나요?
   결과: 필요한 청약통장 종류와 가입 필요 여부를 이해하고 청약 준비가 가능하다.
2. score=8.100 file=주택공급에 관한 규칙.pdf topic=1
   질문: 청약하려면 주택청약종합저축이 꼭 있어야 하나요?
   결과: 가입요건을 충족하면 청약 신청이 가능하고, 이미 사용한 통장은 재사용이 제한된다.
3. score=5.945 file=주택청약 FAQ.pdf topic=4
   질문: 주택을 소유했다고 보지 않는 예외에 해당하면 무주택으로 인정되나요?
   결과: 요건을 충족하면 해당 주택을 소유하지 않은 것으로 보

-- P3 retrieval log 일부 (파싱된 0건) --


-- P4 retrieval log 일부 (파싱된 0건) --




**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 21.083 | 17.973 | 3.11 | 10 | 13.9 |
| P2 | 10.228 | 6.684 | 3.544 | 15 | 22.9 |
| P3 | None | None | None | 0 | 10.3 |
| P4 | None | None | None | 0 | 34.4 |

**문서 겹침도 (Jaccard)**

| 비교 | 겹친 문서 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 2 | 0.50 |
| P1 ∩ P3 | 0 | 0.00 |
| P1 ∩ P4 | 0 | 0.00 |
| P2 ∩ P3 | 0 | 0.00 |
| P2 ∩ P4 | 0 | 0.00 |
| P3 ∩ P4 | 0 | n/a |

In [36]:
metrics2 = compute_metrics(results2, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=51.285 file=청약홈 FAQ - 03.청약가점제 chunk=3
    1월부터 다시 본인의 세대에 합가했는데, 이 경우 부모님은 부양가족으로 인정되나요? A. 직계존속은 신청자와 동일한 주민등록표등본에 3년 이상 ‘연속해서’ 등재되어 있어야하므로 합가한 2019년 1월부터 3년 이상이 경과 해야 비로소 부양가족으로 인정됩니다. 다만, 직계존속과 직계존속의 배우자 중 한 명이라도 주택을 소유(분양권등 포함)하고 있는 경우에는 직계존속과 그 배우자 모두 부양가족
2. score=27.463 file=청약홈 FAQ - 03.청약가점제 chunk=4
   본인과 주민등록표등본상 분리되어 있을 경우 부양가족수에 포함되나요? A. 포함되지 않습니다. 자녀 (부

-- P2 retrieval log 일부 (파싱된 9건) --
## Retrieved Topics
1. score=14.519 file=청약홈 FAQ - 03.청약가점제 topic=2
   질문: 청약가점제에서 부양가족수는 누구를 어떻게 인정하나요?
   결과: 인정되는 부양가족 수에 따라 가점이 산정된다.
2. score=8.332 file=청약홈 FAQ - 03.청약가점제 topic=0
   질문: 청약가점제란 무엇이며 당첨자는 어떻게 선정되나요?
   결과: 가점제 점수에 따라 가점제 배정 물량에서 당첨 여부가 결정된다.
3. score=7.530 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=10
   질문: 민영주택 가점제는 어떤 항목으로 계산되나요?
   결과: 가점 항목을 바탕으로 민영주택 청약가점을 확인할 수 있다.
4. score=6.

-- P3 retrieval log 일부 (파싱된 4건) --
## Seed Phrase Nodes
1. score=8.00 type=Screen label=부양가족
2. score=7.55 type=Screen label=

**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 51.285 | 31.756 | 19.529 | 10 | 13.4 |
| P2 | 14.519 | 8.616 | 5.903 | 9 | 22.0 |
| P3 | 22.09 | 13.32 | 8.77 | 4 | 32.6 |
| P4 | None | None | None | 0 | 67.9 |

**문서 겹침도 (Jaccard)**

| 비교 | 겹친 문서 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 1 | 0.50 |
| P1 ∩ P3 | 2 | 1.00 |
| P1 ∩ P4 | 0 | 0.00 |
| P2 ∩ P3 | 1 | 0.50 |
| P2 ∩ P4 | 0 | 0.00 |
| P3 ∩ P4 | 0 | 0.00 |

In [37]:
metrics3 = compute_metrics(results3, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=23.006 file=청약홈 FAQ - 04.청약 제한 chunk=8
    계약체결일 * 단, 최초로 예비입주자를 입주자로 선정하는 경우로서 동·호수 배정의 추첨에 참가하여 동·호수를 배정받은 경우 : 동·호수배정일 (계약여부와 관계 없이 당첨자 관리) 4. 조합주택(직장, 지역)의 조합원 : 사업계획승인일 5. 조합주택(재건축, 재개발)의 조합원 : 관리처분계획인가일. 단, 2006년 8월 18일 이전 사업승인을 받은 재건축조합원의 경우 ‘사업계획승인일’ 기준으
2. score=22.522 file=주택청약 FAQ.pdf chunk=138
   택세대구성원 요건을 충족하지  않은 것인가요?    「주택공급에 관한 규칙」 부칙 <국토교통부령 제565호, 

-- P2 retrieval log 일부 (파싱된 13건) --
## Retrieved Topics
1. score=14.041 file=청약홈 FAQ - 03.청약가점제 topic=1
   질문: 청약가점제 적용비율은 면적과 지역에 따라 어떻게 달라지나요?
   결과: 해당 주택 유형에서 가점제와 추첨제의 적용 비율을 확인할 수 있다.
2. score=11.016 file=청약홈 FAQ - 01.청약이 잘 안되세요? topic=7
   질문: 수도권 외 거주자인데 1순위가 아닌 2순위로 조회되는 이유는 무엇인가요?
   결과: 실제 청약 주택 기준으로 1순위 여부를 확인할 수 있다.
3. score=10.968 file=청약홈 FAQ - 02.청약통장 topic=5
   질문: 청약통장으로 1순위가 되려면 어떤 기준을 충족해야 하나요?
   결과: 지역·주택유형·납입요

-- P3 retrieval log 일부 (파싱된 5건) --
## Seed Phrase Nodes
1. score=13.86 type=Task label=일반공급 1순위 및 투기과열지구 제한 확인
2. score=13.2

**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 23.006 | 20.626 | 2.38 | 10 | 32.8 |
| P2 | 14.041 | 10.773 | 3.268 | 13 | 39.2 |
| P3 | 13.86 | 9.954 | 3.906 | 5 | 35.7 |
| P4 | None | None | None | 0 | 33.3 |

**문서 겹침도 (Jaccard)**

| 비교 | 겹친 문서 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 2 | 0.67 |
| P1 ∩ P3 | 2 | 0.67 |
| P1 ∩ P4 | 0 | 0.00 |
| P2 ∩ P3 | 2 | 1.00 |
| P2 ∩ P4 | 0 | 0.00 |
| P3 ∩ P4 | 0 | 0.00 |

### 17-e. 질문별 / 파이프라인별 최종 결과 md 저장

`results1/2/3`(파이프라인별 답변 + retrieval log)과 `metrics1/2/3`(score·Jaccard·latency)을
**질문 하나당 한 섹션**으로 묶어 하나의 markdown 파일로 저장합니다.
먼저 17-1~17-d 셀이 모두 실행되어 여섯 변수가 채워져 있어야 합니다.

저장 경로: `PROJECT_DIR/results/three_pipeline_comparison_report.md`

In [38]:
from datetime import datetime

OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "three_pipeline_comparison_report.md"

QUERY_LABELS = {
    1: "질문 1 · 파이프라인 1(Chunk BM25) 지향 — 수치 조건 확인형",
    2: "질문 2 · 파이프라인 2(Topic-based) 지향 — 개인 상황 상담형",
    3: "질문 3 · 파이프라인 3(Graph RAG) 지향 — 규제 패턴 비교형",
}


def _results_to_markdown(query_no, query, results, metrics):
    md = [f"## {QUERY_LABELS.get(query_no, f'질문 {query_no}')}", "",
          f"**질문:** {query}", ""]
    for n, log, answer, elapsed, code in results:
        status = "성공" if code == 0 else "실패"
        md += [f"### {PIPELINE_TITLES[n]} ({status}, {elapsed:.1f}s)", "",
               answer or "(답변 없음)", ""]
        if log:
            md += ["<details><summary>retrieval log</summary>", "", "```",
                   log[:6000], "```", "", "</details>", ""]
    score_table, overlap_table = _metrics_to_markdown(metrics)
    md += ["### 성능 지표", "", "**Score 분포 / 확신도**", "", score_table, "",
           "**문서 겹침도 (Jaccard)**", "", overlap_table, "", "---", ""]
    return "\n".join(md)


def build_comparison_report(queries, results_list, metrics_list, output_path=REPORT_PATH):
    """질문/결과/성능지표를 받아 하나의 .md 리포트로 저장."""
    sections = [
        "# RAG 파이프라인 비교 리포트 (청약 시장 Q&A)", "",
        f"생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", "",
        "| | 파이프라인 | 방식 |\n|---|---|---|\n"
        "| P1 | Chunk BM25 (baseline) | 원문 chunk 검색 |\n"
        "| P2 | Topic-based (structural) | 토픽/요건 구조화 검색 |\n"
        "| P3 | Graph RAG (relational) | 토픽 그래프 기반 패턴 검색 |\n"
        "| P4 | Search Agent (multi-step) | agent가 스스로 다중 검색 후 답변 |", "",
        "---", "",
    ]
    for i, (query, results, metrics) in enumerate(zip(queries, results_list, metrics_list), start=1):
        sections.append(_results_to_markdown(i, query, results, metrics))
    report_text = "\n".join(sections)
    output_path.write_text(report_text, encoding="utf-8")
    print(f"리포트 저장 완료: {output_path}  ({len(report_text)} chars)")
    return output_path


report_path = build_comparison_report(
    queries=[Q1, Q2, Q3],
    results_list=[results1, results2, results3],
    metrics_list=[metrics1, metrics2, metrics3],
)

리포트 저장 완료: D:\snu_python\graphrag\results\three_pipeline_comparison_report.md  (39876 chars)
